# Fish2 Raphe superior soma-in-ROI — plotting/reclustering from cached NBLAST — cleaned check-ID workflow + true lateral static views

This notebook is **plotting / clustering-exploration only** for the new Raphe-superior soma-in-ROI run.

It does **not** recompute NBLAST. It reads the cached NBLAST distance matrix from the full run, then:

1. tries multiple hierarchical clustering settings with fewer cluster numbers;
2. tries multiple UMAP settings from the same precomputed NBLAST distance matrix to improve visual separation;
3. lets you select one active clustering + one active UMAP embedding for all downstream cells;
4. plots soma/root locations of the neurons used inside the clean transparent brain shell;
5. keeps the same clean 3D shell cluster-rendering / PDF-export machinery as the latest notebooks.


**v26 cleanup:** fixes active UMAP handling, defines all cluster helper functions before use, adds `CELLS_TO_CHECK` cluster/UMAP/brain-location plots, and removes the fragile MDS call.

**v28 update:** all fixed/static 3D views now use the same camera conventions: `top`, true lateral `side` (left/right, looking along the y-axis), optional `coronal` (old x-axis view), and `angled`. A cluster-colored soma-location static 3D figure cell is included after the soma-location cell.

In [ ]:
# ============================================================
# 1. Imports + configuration
# ============================================================

from pathlib import Path
from collections import defaultdict
import os
import re
import json
import math
import warnings

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from IPython.display import display, HTML, clear_output
from tqdm import tqdm

# Plotly for interactive cluster rendering
try:
    import plotly.graph_objects as go
    HAS_PLOTLY = True
except Exception as e:
    HAS_PLOTLY = False
    print("Plotly not available:", repr(e))

# Optional widgets for cluster selector
try:
    import ipywidgets as widgets
    HAS_WIDGETS = True
except Exception as e:
    HAS_WIDGETS = False
    print("ipywidgets not available:", repr(e))

# Optional neuPrint soma metadata fetching.
try:
    from neuprint import Client, fetch_neurons, NeuronCriteria as NC
    HAS_NEUPRINT = True
except Exception as e:
    HAS_NEUPRINT = False
    print("neuprint-python not available:", repr(e))

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
SKELETON_DIR = PROJECT_DIR / "skeletons"

# This should be the output folder from your previous working v14 notebook.
PREVIOUS_OUT_ROOT = PROJECT_DIR / "fish2_raphe_morphology_v20_rapheSuperior_somaInROI_outputs"

# This notebook writes only new/updated plots and tables here.
PLOT_OUT_ROOT = PREVIOUS_OUT_ROOT / "plotting_v28_recluster_umap_checkids_lateralStaticViews"
PLOT_OUT_ROOT.mkdir(parents=True, exist_ok=True)

# Canonical dataset key for this notebook.
# Defined here so earlier sanity-check/plotting cells do not depend on later figure-ready config cells.
FIGURE_DATASET_KEY = "raphe_superior_soma_in_roi"

# Cells you want to track in the active soma-in-Raphe-superior clustering.
# These will be checked against the active assignments, highlighted on UMAP,
# and plotted in the brain shell if their SWCs are available.
CELLS_TO_CHECK = [
    100791363,
    102967931,
    106511646,
    105165440,
    100427087,
    100312010,
    103164598,
    103159060,
    101079563,
    103160069,
    100098958,
    100668438,
    100528051,
    100463138,
    109988161,
]

# Shared figure-ready output folder; later figure-ready cells reuse this same path.
FIGURE_READY_DIR = PLOT_OUT_ROOT / "figure_ready_reclustered_fixed_3d_shell_soma_in_roi"
FIGURE_READY_DIR.mkdir(parents=True, exist_ok=True)

SELECTED_RAW_SWC_DIR = SKELETON_DIR / "selected_raw_swc"
RAPHE_SUPERIOR_SOMA_IN_ROI_RAW_SWC_DIR = SKELETON_DIR / "raphe_superior_soma_in_roi_raw_swc"

# Dataset keys from v14.
DATASET_CONFIG = {
    "raphe_superior_soma_in_roi": {
        "label": "Raphe superior soma-in-ROI neurons",
        "raw_swc_dir": SKELETON_DIR / "raphe_superior_soma_in_roi_raw_swc",
    },
}

# Figure export settings.
FIG_DPI = 300
SAVE_PNG = True
SAVE_PDF = True
SAVE_SVG = True
SAVE_INTERACTIVE_HTML = True

# Rendering options.
SHOW_BRAIN_SHELL = True
SHELL_ALPHA = 0.035
CELL_LINE_WIDTH = 2.0  # Tune this: smaller values make crowded clusters easier to inspect
CONTEXT_LINE_WIDTH = 0.45
POPULATION_CONTEXT_ALPHA = 0.020
MAX_CONTEXT_CELLS = 250
MAX_CELLS_PER_CLUSTER_RENDER = 120

# v18 cluster-view modes.
# "all_three" displays three separate brains: skeletons-only, somas-only, and combined.
# Other options display one brain.
DEFAULT_CLUSTER_RENDER_MODE = "all_three"
CLUSTER_RENDER_MODE_OPTIONS = ["all_three", "skeletons_only", "somas_only", "combined"]
SAVE_EACH_RENDER_MODE_HTML = True


# Display-only orientation correction.
# This flips the shell, neurons, and somas together, only for plotting.
# It does not alter clusters, SWCs, UMAPs, or CSV exports.
RENDER_FLIP_X = False
RENDER_FLIP_Y = False
RENDER_FLIP_Z = True


# Shared fixed-camera settings for ALL static 3D exports.
# In this Fish2 rendering, the old "side" camera looked along x and therefore
# produced a coronal/front-back view. Here, "side" is a true left/right lateral
# view looking along y. Use "coronal" if you want the old x-axis view.
STATIC_3D_SIDE_DIRECTION = "left"  # "left" or "right"
STATIC_3D_CAMERA_DISTANCE = 2.35
STATIC_3D_ANGLED_EYE = dict(x=1.55, y=1.65, z=1.15)


def camera_for_static_3d_view(view, side_direction=None, distance=None):
    """
    Central camera helper used by all fixed/static 3D Plotly views.

    Views:
      - "top": dorsal/top view, looking along z.
      - "side": true left/right lateral view, looking along y.
      - "left": lateral view from +y.
      - "right": lateral view from -y.
      - "coronal": old x-axis view, useful if you want front/back/coronal.
      - "angled": 3/4 perspective.
    """
    view = str(view).lower()
    side_direction = side_direction or globals().get("STATIC_3D_SIDE_DIRECTION", "left")
    distance = float(distance or globals().get("STATIC_3D_CAMERA_DISTANCE", 2.35))

    if view == "top":
        return dict(
            eye=dict(x=0.0, y=0.0, z=distance),
            up=dict(x=0, y=1, z=0),
        )

    if view in ("side", "left", "right"):
        if view == "left":
            y_eye = distance
        elif view == "right":
            y_eye = -distance
        else:
            y_eye = distance if side_direction == "left" else -distance
        return dict(
            eye=dict(x=0.0, y=y_eye, z=0.0),
            up=dict(x=0, y=0, z=1),
        )

    if view == "coronal":
        # This is the old "side" camera from earlier notebook versions.
        return dict(
            eye=dict(x=distance, y=0.0, z=0.0),
            up=dict(x=0, y=0, z=1),
        )

    if view == "front":
        # Alias kept for backwards compatibility, but lateral side is preferred.
        return dict(
            eye=dict(x=0.0, y=distance, z=0.0),
            up=dict(x=0, y=0, z=1),
        )

    if view == "angled":
        angled = globals().get("STATIC_3D_ANGLED_EYE", dict(x=1.55, y=1.65, z=1.15))
        return dict(
            eye=angled,
            up=dict(x=0, y=0, z=1),
        )

    raise ValueError(f"Unknown static 3D view: {view}")

# Real soma/cell-body rendering.
USE_NEUPRINT_SOMAS = True
NEUPRINT_SERVER = "https://neuprint-fish2.janelia.org"
NEUPRINT_DATASET = "fish2"
NEUPRINT_TOKEN_FALLBACK = ""  # Prefer env var NEUPRINT_TOKEN or local neuprint_token.txt
SOMA_LOCATION_PRIORITY = ["somaLocation", "tosomaLocation", "rootLocation"]
# Soma/cell-body display mode:
#   "swc_local_blob" = preferred: render local SWC nodes/edges around soma/root with radius-scaled markers.
#   "sphere"         = metadata somaLocation/somaRadius as a simple sphere.
#   "marker"         = metadata/root as a simple point.
SOMA_RENDER_MODE = "swc_radius_mesh"
SOMA_MARKER_SIZE = 5
SOMA_SPHERE_OPACITY = 0.9
SOMA_SPHERE_RADIUS_SCALE = 1.0
SOMA_MARKER_SIZE_MIN = 4
SOMA_MARKER_SIZE_MAX = 12
SOMA_SPHERE_THETA_STEPS = 10
SOMA_SPHERE_PHI_STEPS = 8
FORCE_REFETCH_SOMAS = False

# Skeleton display settings.
# "lines" is fast and clean. "radius_tubes" is experimental and much heavier.
CELL_RENDER_MODE = "lines"
CELL_LINE_ALPHA = 0.95
CELL_TUBE_RADIUS_SCALE = 0.45
CELL_TUBE_MIN_RADIUS = 8.0
CELL_TUBE_MAX_RADIUS = 35.0
CELL_TUBE_MAX_EDGES_PER_NEURON = 1200
CELL_TUBE_SIDES = 6

# Soma/cell-body display settings.
# "swc_radius_mesh" is default: render a small surface mesh from the local SWC
# nodes/radii around the neuPrint soma/root anchor. This is the most realistic
# soma geometry available from neuPrint/SWC without external segmentation meshes.
# "local_body_mesh" tries body_meshes/<bodyId>.obj/.ply and clips near soma.
# "marker" and "sphere" are simple fallbacks.
SOMA_RADIUS_MESH_GRAPH_HOPS = 5
SOMA_RADIUS_MESH_EUCLIDEAN_RADIUS = 650.0
SOMA_RADIUS_MESH_MAX_NODES = 22
SOMA_RADIUS_MESH_MIN_NODES = 4
SOMA_RADIUS_MESH_RADIUS_SCALE = 1.15
SOMA_RADIUS_MESH_MIN_RADIUS = 25.0
SOMA_RADIUS_MESH_MAX_RADIUS = 230.0
SOMA_RADIUS_MESH_SPHERE_THETA = 9
SOMA_RADIUS_MESH_SPHERE_PHI = 7
SOMA_RADIUS_MESH_CYLINDER_SIDES = 9
SOMA_RADIUS_MESH_OPACITY = 0.92

# Optional true segmentation/body mesh hook.
# If you export per-body meshes later, place them in body_meshes/ as <bodyId>.obj or <bodyId>.ply.
# The notebook will clip vertices/faces near somaLocation/root and render that soma-region mesh.
LOCAL_BODY_MESH_DIR = PROJECT_DIR / "body_meshes"
PREFER_LOCAL_BODY_MESH_SOMA = True
LOCAL_BODY_MESH_SOMA_CLIP_RADIUS = 1200.0
LOCAL_BODY_MESH_MAX_FACES = 8000
LOCAL_BODY_MESH_OPACITY = 0.82

# Cluster colors: every cell and soma in a displayed cluster uses the same color.
CLUSTER_COLOR_PALETTE = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd",
    "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf",
    "#393b79", "#637939", "#8c6d31", "#843c39", "#7b4173",
    "#3182bd", "#31a354", "#756bb1", "#636363", "#e6550d",
]

# Output folders.
FIG_DIR = PLOT_OUT_ROOT / "figures"
RENDER_DIR = PLOT_OUT_ROOT / "cluster_renderings"
EXPORT_DIR = PLOT_OUT_ROOT / "cluster_id_exports"
SOMA_DIR = PLOT_OUT_ROOT / "neuprint_soma_metadata"
for p in [FIG_DIR, RENDER_DIR, EXPORT_DIR, SOMA_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Previous outputs:", PREVIOUS_OUT_ROOT)
print("New plot-only outputs:", PLOT_OUT_ROOT)
print("Render flips: X=", RENDER_FLIP_X, "Y=", RENDER_FLIP_Y, "Z=", RENDER_FLIP_Z)


In [ ]:
# ============================================================
# 2. Load previous v14 clustering outputs
# ============================================================

def require_file(path, description="file"):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {description}: {path}\n"
            "This plotting-only notebook expects that the v14 notebook has already completed."
        )
    return path


def load_dataset_from_previous_outputs(dataset_key, cfg):
    ds = dict(cfg)
    # Prefer the root assignment file, then fall back to cluster_id_exports.
    assign_path = PREVIOUS_OUT_ROOT / f"{dataset_key}_cluster_assignments.csv"
    if not assign_path.exists():
        assign_path = PREVIOUS_OUT_ROOT / "cluster_id_exports" / f"{dataset_key}_per_neuron_cluster_assignments.csv"
    require_file(assign_path, f"{dataset_key} assignments")
    assignments = pd.read_csv(assign_path)
    assignments["bodyId"] = assignments["bodyId"].astype(int)
    assignments["cluster"] = assignments["cluster"].astype(int)
    if "cluster_size" not in assignments.columns:
        sizes = assignments["cluster"].value_counts()
        assignments["cluster_size"] = assignments["cluster"].map(sizes).astype(int)
    if "cluster_name" not in assignments.columns:
        assignments["cluster_name"] = assignments["cluster"].map(lambda x: f"{dataset_key}_C{int(x):02d}")
    ds["assignments"] = assignments
    ds["body_ids"] = assignments["bodyId"].astype(int).tolist()
    ds["assignment_path"] = assign_path

    emb_path = PREVIOUS_OUT_ROOT / f"{dataset_key}_embeddings.csv"
    if emb_path.exists():
        emb = pd.read_csv(emb_path)
        emb["bodyId"] = emb["bodyId"].astype(int)
        ds["embeddings"] = emb
        ds["embedding_path"] = emb_path
    else:
        print(f"No embedding file found for {dataset_key}: {emb_path}")

    sweep_path = PREVIOUS_OUT_ROOT / f"{dataset_key}_cluster_sweep.csv"
    if sweep_path.exists():
        ds["sweep"] = pd.read_csv(sweep_path)

    return ds


datasets = {}
for key, cfg in DATASET_CONFIG.items():
    try:
        datasets[key] = load_dataset_from_previous_outputs(key, cfg)
        a = datasets[key]["assignments"]
        print(f"Loaded {key}: {len(a)} cells, {a['cluster'].nunique()} clusters from {datasets[key]['assignment_path']}")
    except FileNotFoundError as e:
        print(str(e))

if not datasets:
    raise RuntimeError("No previous dataset outputs could be loaded.")

# Quick display of cluster sizes.
for key, ds in datasets.items():
    print("\n", "="*80)
    print(ds["label"])
    display(ds["assignments"].groupby("cluster").size().sort_values(ascending=False).to_frame("n_cells").head(30))


In [ ]:
# ============================================================
# 2b. Recluster + recompute UMAPs from cached NBLAST distance
# ============================================================
# This cell does NOT recompute NBLAST. It reads the cached NBLAST distance
# matrix from the soma-in-Raphe-superior run, then explores:
#   - fewer cluster numbers
#   - different linkage methods
#   - different UMAP parameters using metric='precomputed'
#
# After inspecting the figures, change ACTIVE_* below and rerun this cell.
# Downstream cells use datasets[DATASET_KEY]['assignments'] and
# datasets[DATASET_KEY]['embeddings']['umap'].

from scipy.cluster.hierarchy import linkage, cut_tree
from scipy.spatial.distance import squareform
from sklearn.metrics import silhouette_score

try:
    import umap
    HAS_UMAP_RECLUSTER = True
except Exception as e:
    HAS_UMAP_RECLUSTER = False
    print('umap-learn not available:', repr(e))

DATASET_KEY = FIGURE_DATASET_KEY

# Cluster cuts to compare. Keep this modest; the point is to reduce the auto-31 overfragmentation.
CLUSTER_K_VALUES = [6, 8, 10, 12, 14, 16, 18]
LINKAGE_METHODS_TO_TRY = ['complete', 'average', 'weighted']

# UMAP settings. Lower min_dist tightens islands; lower n_neighbors emphasizes local structure.
# Your current good-looking example was complete k=12 on nn40_md005.
UMAP_PARAM_GRID = [
    {'name': 'nn10_md000', 'n_neighbors': 10, 'min_dist': 0.00, 'spread': 1.0},
    {'name': 'nn15_md000', 'n_neighbors': 15, 'min_dist': 0.00, 'spread': 1.0},
    {'name': 'nn20_md000', 'n_neighbors': 20, 'min_dist': 0.00, 'spread': 1.0},
    {'name': 'nn25_md001', 'n_neighbors': 25, 'min_dist': 0.01, 'spread': 1.0},
    {'name': 'nn30_md001', 'n_neighbors': 30, 'min_dist': 0.01, 'spread': 1.0},
    {'name': 'nn40_md005', 'n_neighbors': 40, 'min_dist': 0.05, 'spread': 1.0},
]

# Choose the active solution here after inspecting the saved grids.
ACTIVE_LINKAGE_METHOD = 'complete'
ACTIVE_CLUSTER_K = 12
ACTIVE_UMAP_NAME = 'nn40_md005'
ACTIVE_RANDOM_STATE = 7

RECLUSTER_DIR = PLOT_OUT_ROOT / 'recluster_from_cached_nblast'
RECLUSTER_ASSIGN_DIR = RECLUSTER_DIR / 'assignments'
RECLUSTER_EMB_DIR = RECLUSTER_DIR / 'embeddings'
RECLUSTER_FIG_DIR = RECLUSTER_DIR / 'figures'
for _p in [RECLUSTER_DIR, RECLUSTER_ASSIGN_DIR, RECLUSTER_EMB_DIR, RECLUSTER_FIG_DIR]:
    _p.mkdir(parents=True, exist_ok=True)


# Stable cluster colors for this reclustering cell. The later figure-ready cell
# defines the same helper, but this cell runs before that in a clean kernel.
def get_cluster_color(dataset_key, cluster):
    cluster = int(cluster)
    return CLUSTER_COLOR_PALETTE[(cluster - 1) % len(CLUSTER_COLOR_PALETTE)]



def _find_cached_nblast_distance(dataset_key=DATASET_KEY):
    """Find the cached distance matrix from the full soma-in-ROI run."""
    search_dirs = [
        PREVIOUS_OUT_ROOT / 'nblast_cache' / dataset_key,
        PREVIOUS_OUT_ROOT / 'nblast_cache',
        PREVIOUS_OUT_ROOT,
    ]
    patterns = [
        f'{dataset_key}*distance*.pkl',
        f'{dataset_key}*dist*.pkl',
        f'*{dataset_key}*distance*.pkl',
        f'*{dataset_key}*dist*.pkl',
        f'{dataset_key}*distance*.csv',
        f'{dataset_key}*dist*.csv',
        f'*{dataset_key}*distance*.csv',
        f'*{dataset_key}*dist*.csv',
    ]
    hits = []
    for d in search_dirs:
        if not d.exists():
            continue
        for pat in patterns:
            hits.extend(sorted(d.glob(pat)))
            hits.extend(sorted(d.glob(f'**/{pat}')))

    hits_unique, seen = [], set()
    for h in hits:
        hp = str(h.resolve())
        if hp not in seen and h.is_file():
            seen.add(hp)
            hits_unique.append(h)

    if not hits_unique:
        raise FileNotFoundError(
            f'Could not find cached NBLAST distance matrix for {dataset_key} under {PREVIOUS_OUT_ROOT}.\n'
            'Run the full soma-in-ROI notebook through NBLAST first.'
        )

    hits_unique = sorted(hits_unique, key=lambda p: (0 if 'rank' in p.name.lower() else 1, len(str(p))))
    print('Candidate distance matrices:')
    for h in hits_unique[:10]:
        print('  ', h)
    return hits_unique[0]


def load_cached_nblast_distance(dataset_key=DATASET_KEY):
    path = _find_cached_nblast_distance(dataset_key)
    print('\nLoading distance matrix:', path)
    if path.suffix.lower() == '.pkl':
        dist = pd.read_pickle(path)
    else:
        dist = pd.read_csv(path, index_col=0)

    dist.index = dist.index.astype(int)
    dist.columns = dist.columns.astype(int)

    loaded_ids = datasets[dataset_key]['assignments']['bodyId'].astype(int).tolist()
    keep = [bid for bid in loaded_ids if bid in dist.index and bid in dist.columns]
    missing = sorted(set(loaded_ids) - set(keep))
    if missing:
        print(f'Warning: {len(missing)} loaded cells missing from distance matrix. First:', missing[:20])

    dist = dist.loc[keep, keep].astype(float)
    dist = (dist + dist.T) / 2.0
    np.fill_diagonal(dist.values, 0.0)
    dist[dist < 0] = 0.0
    print(f'Distance matrix used for reclustering: {dist.shape[0]} x {dist.shape[1]}')
    return dist, path


def make_linkage(dist, method):
    arr = dist.to_numpy(float)
    arr = (arr + arr.T) / 2.0
    np.fill_diagonal(arr, 0.0)
    return linkage(squareform(arr, checks=False), method=method)


def assignment_from_linkage(dist, Z, k, method, dataset_key=DATASET_KEY):
    ids = np.array(dist.index.astype(int).tolist())
    labels = cut_tree(Z, n_clusters=int(k)).ravel().astype(int) + 1
    a = pd.DataFrame({'bodyId': ids, 'cluster_raw': labels})

    # Renumber by size so C01 is largest.
    order = a['cluster_raw'].value_counts().sort_values(ascending=False).index.tolist()
    raw_to_new = {raw: i + 1 for i, raw in enumerate(order)}
    a['cluster'] = a['cluster_raw'].map(raw_to_new).astype(int)
    sizes = a['cluster'].value_counts()
    a['cluster_size'] = a['cluster'].map(sizes).astype(int)
    a['cluster_name'] = a['cluster'].map(lambda c: f'{dataset_key}_{method}_k{int(k):02d}_C{int(c):02d}')
    a['linkage_method'] = method
    a['recluster_k'] = int(k)
    return a[['bodyId','cluster','cluster_size','cluster_name','linkage_method','recluster_k']].sort_values(['cluster','bodyId']).reset_index(drop=True)


def compute_umap_from_distance(dist, params, random_state=ACTIVE_RANDOM_STATE):
    if not HAS_UMAP_RECLUSTER:
        raise RuntimeError('umap-learn is required for recomputing UMAP embeddings.')
    reducer = umap.UMAP(
        n_components=2,
        metric='precomputed',
        n_neighbors=int(params['n_neighbors']),
        min_dist=float(params['min_dist']),
        spread=float(params.get('spread', 1.0)),
        random_state=int(random_state),
    )
    coords = reducer.fit_transform(dist.to_numpy(float))
    emb = pd.DataFrame({
        'bodyId': dist.index.astype(int).tolist(),
        'umap1': coords[:, 0],
        'umap2': coords[:, 1],
        'umap_name': params['name'],
        'umap_n_neighbors': int(params['n_neighbors']),
        'umap_min_dist': float(params['min_dist']),
        'umap_spread': float(params.get('spread', 1.0)),
        'umap_random_state': int(random_state),
    })
    return emb


def _save_current_fig(fig, out_base, dpi=350):
    out_base = Path(out_base)
    out_base.parent.mkdir(parents=True, exist_ok=True)
    if SAVE_PNG:
        fig.savefig(out_base.with_suffix('.png'), dpi=dpi, bbox_inches='tight')
    if SAVE_PDF:
        fig.savefig(out_base.with_suffix('.pdf'), bbox_inches='tight')
    if SAVE_SVG:
        fig.savefig(out_base.with_suffix('.svg'), bbox_inches='tight')


def plot_umap_assignment(emb, assignments, title, out_base=None, point_size=28, legend=True, highlight_ids=None):
    df = emb.merge(assignments, on='bodyId', how='inner')
    clusters = sorted(df['cluster'].astype(int).unique())
    fig, ax = plt.subplots(figsize=(7.2, 6.2), dpi=320)
    for cl in clusters:
        sub = df[df['cluster'].astype(int) == cl]
        ax.scatter(
            sub['umap1'], sub['umap2'],
            s=point_size,
            color=get_cluster_color(DATASET_KEY, cl) if 'get_cluster_color' in globals() else None,
            alpha=0.90,
            edgecolor='white',
            linewidth=0.35,
            label=f'C{cl:02d} (n={len(sub)})',
        )

    if highlight_ids:
        highlight_ids = sorted(set(int(x) for x in highlight_ids))
        h = df[df['bodyId'].astype(int).isin(highlight_ids)]
        if len(h):
            ax.scatter(h['umap1'], h['umap2'], s=160, marker='*', color='yellow', edgecolor='black', linewidth=0.8, zorder=10, label='CELLS_TO_CHECK')
            for _, r in h.iterrows():
                ax.text(r['umap1'], r['umap2'], str(int(r['bodyId'])), fontsize=6, color='black', zorder=11)
        missing = sorted(set(highlight_ids) - set(h['bodyId'].astype(int).tolist()))
        if missing:
            print('Highlighted IDs not found in active embedding:', missing)

    ax.set_title(title)
    ax.set_xlabel('umap1')
    ax.set_ylabel('umap2')
    ax.set_aspect('equal', adjustable='datalim')
    if legend and len(clusters) <= 18:
        ax.legend(frameon=False, fontsize=6.5, loc='best', markerscale=0.9)
    fig.tight_layout()
    if out_base is not None:
        _save_current_fig(fig, out_base)
        print('Saved:', out_base)
    display(fig)
    plt.close(fig)
    return df


def plot_k_grid_for_method(emb, assignment_variants, method, k_values=CLUSTER_K_VALUES):
    n = len(k_values)
    ncols = 4
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.1*ncols, 3.7*nrows), dpi=260)
    axes = np.asarray(axes).ravel()
    for ax, k in zip(axes, k_values):
        a = assignment_variants[(method, int(k))]
        df = emb.merge(a, on='bodyId', how='inner')
        for cl in sorted(df['cluster'].astype(int).unique()):
            sub = df[df['cluster'].astype(int) == cl]
            ax.scatter(sub['umap1'], sub['umap2'], s=10, alpha=0.85, color=get_cluster_color(DATASET_KEY, cl) if 'get_cluster_color' in globals() else None, linewidth=0)
        ax.set_title(f'{method}, k={k}')
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_aspect('equal', adjustable='datalim')
    for ax in axes[n:]:
        ax.axis('off')
    fig.suptitle(f'Raphe superior soma-in-ROI | {method} linkage k sweep | UMAP {emb["umap_name"].iloc[0]}', y=1.02, fontsize=16)
    fig.tight_layout()
    out_base = RECLUSTER_FIG_DIR / f'grid_{method}_ks_on_umap_{emb["umap_name"].iloc[0]}'
    _save_current_fig(fig, out_base)
    display(fig)
    plt.close(fig)


def run_reclustering_and_umap_exploration():
    dist, dist_path = load_cached_nblast_distance(DATASET_KEY)

    # 1) Cluster variants.
    assignment_variants = {}
    summary_rows = []
    for method in LINKAGE_METHODS_TO_TRY:
        print('\nLinkage:', method)
        Z = make_linkage(dist, method)
        for k in CLUSTER_K_VALUES:
            a = assignment_from_linkage(dist, Z, int(k), method)
            assignment_variants[(method, int(k))] = a
            out_csv = RECLUSTER_ASSIGN_DIR / f'{DATASET_KEY}_{method}_k{int(k):02d}_assignments.csv'
            a.to_csv(out_csv, index=False)
            sizes = a['cluster'].value_counts()
            labels_aligned = a.set_index('bodyId').loc[dist.index.astype(int).tolist(), 'cluster'].to_numpy()
            try:
                sil = silhouette_score(dist.to_numpy(float), labels_aligned, metric='precomputed')
            except Exception:
                sil = np.nan
            summary_rows.append({
                'linkage_method': method,
                'k': int(k),
                'n_cells': len(a),
                'largest_cluster': int(sizes.max()),
                'smallest_cluster': int(sizes.min()),
                'n_singletons': int((sizes == 1).sum()),
                'n_clusters_ge_5': int((sizes >= 5).sum()),
                'silhouette_precomputed_distance': sil,
                'assignments_csv': str(out_csv),
            })
    summary = pd.DataFrame(summary_rows)
    summary_path = RECLUSTER_DIR / 'clustering_method_k_summary.csv'
    summary.to_csv(summary_path, index=False)
    print('\nSaved clustering summary:', summary_path)
    display(summary.sort_values(['linkage_method','k']))

    # 2) UMAP variants.
    embeddings = {}
    for params in UMAP_PARAM_GRID:
        print('\nComputing UMAP:', params)
        emb = compute_umap_from_distance(dist, params)
        embeddings[params['name']] = emb
        emb_path = RECLUSTER_EMB_DIR / f'{DATASET_KEY}_umap_{params["name"]}.csv'
        emb.to_csv(emb_path, index=False)
        print('Saved embedding:', emb_path)

    # 3) Plot active clustering on all UMAP variants, with CELLS_TO_CHECK highlighted.
    active_a = assignment_variants[(ACTIVE_LINKAGE_METHOD, int(ACTIVE_CLUSTER_K))]
    for name, emb in embeddings.items():
        plot_umap_assignment(
            emb, active_a,
            title=f'Raphe superior soma-in-ROI | {ACTIVE_LINKAGE_METHOD} k={ACTIVE_CLUSTER_K} | UMAP {name}',
            out_base=RECLUSTER_FIG_DIR / f'active_{ACTIVE_LINKAGE_METHOD}_k{ACTIVE_CLUSTER_K:02d}_on_umap_{name}',
            legend=(ACTIVE_CLUSTER_K <= 16),
            highlight_ids=CELLS_TO_CHECK,
        )

    # 4) K sweep grids on active UMAP.
    active_emb = embeddings[ACTIVE_UMAP_NAME]
    for method in LINKAGE_METHODS_TO_TRY:
        plot_k_grid_for_method(active_emb, assignment_variants, method, CLUSTER_K_VALUES)

    return dist, assignment_variants, embeddings, summary


def apply_active_recluster_solution():
    """Overwrite datasets[DATASET_KEY] assignments/embeddings so all downstream cells use your choice."""
    if 'touching_recluster_assignment_variants' not in globals() or 'touching_recluster_embeddings' not in globals():
        raise RuntimeError('Run run_reclustering_and_umap_exploration() first.')

    key = (ACTIVE_LINKAGE_METHOD, int(ACTIVE_CLUSTER_K))
    if key not in touching_recluster_assignment_variants:
        raise KeyError(f'Missing active assignment variant {key}; available: {list(touching_recluster_assignment_variants.keys())[:10]} ...')
    if ACTIVE_UMAP_NAME not in touching_recluster_embeddings:
        raise KeyError(f'Missing active UMAP {ACTIVE_UMAP_NAME}; available: {list(touching_recluster_embeddings.keys())}')

    a = touching_recluster_assignment_variants[key].copy()
    emb = touching_recluster_embeddings[ACTIVE_UMAP_NAME].copy()

    datasets[DATASET_KEY]['assignments_original_auto31'] = datasets[DATASET_KEY]['assignments'].copy()
    if 'embeddings' in datasets[DATASET_KEY]:
        datasets[DATASET_KEY]['embeddings_original_auto31'] = datasets[DATASET_KEY]['embeddings']

    datasets[DATASET_KEY]['assignments'] = a
    # Store embeddings in a dictionary so figure-ready cells can explicitly request 'umap'.
    datasets[DATASET_KEY]['embeddings'] = {'umap': emb}
    datasets[DATASET_KEY]['active_embedding_df'] = emb
    datasets[DATASET_KEY]['label'] = f'Raphe superior soma-in-ROI | {ACTIVE_LINKAGE_METHOD} k={ACTIVE_CLUSTER_K} | UMAP {ACTIVE_UMAP_NAME}'
    datasets[DATASET_KEY]['active_linkage_method'] = ACTIVE_LINKAGE_METHOD
    datasets[DATASET_KEY]['active_k'] = int(ACTIVE_CLUSTER_K)
    datasets[DATASET_KEY]['active_umap_name'] = ACTIVE_UMAP_NAME

    active_assign_path = RECLUSTER_DIR / 'ACTIVE_assignments.csv'
    active_emb_path = RECLUSTER_DIR / 'ACTIVE_umap_embedding.csv'
    a.to_csv(active_assign_path, index=False)
    emb.to_csv(active_emb_path, index=False)
    print('='*80)
    print('ACTIVE reclustering applied to downstream notebook state')
    print('Dataset:', DATASET_KEY)
    print('Linkage:', ACTIVE_LINKAGE_METHOD)
    print('k:', ACTIVE_CLUSTER_K)
    print('UMAP:', ACTIVE_UMAP_NAME)
    print('Saved:', active_assign_path)
    print('Saved:', active_emb_path)
    print('='*80)
    display(a.groupby('cluster').size().rename('n_cells').reset_index())
    return a, emb


touching_recluster_dist, touching_recluster_assignment_variants, touching_recluster_embeddings, touching_recluster_summary = run_reclustering_and_umap_exploration()
touching_active_assignments, touching_active_embeddings = apply_active_recluster_solution()


In [ ]:
# ============================================================
# 3. Active UMAP figure from reclustered embedding
# ============================================================
# IMPORTANT:
# After Cell 2b, embeddings are stored as:
#     datasets[FIGURE_DATASET_KEY]["embeddings"] = {"umap": <DataFrame>}
# Older notebooks stored embeddings directly as a DataFrame.
# This cell handles both formats and does NOT try to plot MDS.

import matplotlib.colors as mcolors


def savefig_all(fig, path_base):
    path_base = Path(path_base)
    path_base.parent.mkdir(parents=True, exist_ok=True)
    if SAVE_PNG:
        fig.savefig(path_base.with_suffix(".png"), dpi=FIG_DPI, bbox_inches="tight")
    if SAVE_PDF:
        fig.savefig(path_base.with_suffix(".pdf"), bbox_inches="tight")
    if SAVE_SVG:
        fig.savefig(path_base.with_suffix(".svg"), bbox_inches="tight")


def get_cluster_color(dataset_key, cluster):
    """Stable cluster color used across UMAPs and 3D cluster renders."""
    cluster = int(cluster)
    return CLUSTER_COLOR_PALETTE[(cluster - 1) % len(CLUSTER_COLOR_PALETTE)]


def get_cluster_color_map(dataset_key, clusters):
    clusters = sorted([int(c) for c in pd.Series(clusters).dropna().unique()])
    return {cl: get_cluster_color(dataset_key, cl) for cl in clusters}


def get_embedding_dataframe(ds, method="umap"):
    """
    Return an embedding DataFrame from either the new dict-style storage or
    older DataFrame-style storage.

    New format after reclustering:
        ds["embeddings"] = {"umap": df}

    Old format from the original run:
        ds["embeddings"] = df
    """
    emb_obj = ds.get("embeddings", None)
    emb = None

    if isinstance(emb_obj, dict):
        emb = emb_obj.get(method, None)
    elif isinstance(emb_obj, pd.DataFrame):
        emb = emb_obj

    if emb is None and method == "umap" and "active_embedding_df" in ds:
        emb = ds["active_embedding_df"]

    if emb is None:
        return None

    emb = emb.copy()
    if "bodyId" not in emb.columns:
        raise RuntimeError("Embedding table exists but has no bodyId column.")
    emb["bodyId"] = emb["bodyId"].astype(int)
    return emb


def plot_active_umap_from_dataset(dataset_key=FIGURE_DATASET_KEY, label_points=False):
    """
    Plot the active UMAP using the active clustering assignment table.
    This is the figure you should use for the current chosen reclustering.
    """
    if dataset_key not in datasets:
        raise KeyError(f"Dataset {dataset_key!r} not found. Available: {list(datasets.keys())}")

    ds = datasets[dataset_key]
    emb = get_embedding_dataframe(ds, "umap")
    if emb is None:
        print(f"Skipping {dataset_key}: active UMAP embedding not available. Run Cell 2b first.")
        return None

    xcol, ycol = "umap1", "umap2"
    if xcol not in emb.columns or ycol not in emb.columns:
        print(f"Skipping {dataset_key}: columns {xcol}/{ycol} not found in active embedding.")
        print("Embedding columns are:", list(emb.columns))
        return None

    a = ds["assignments"].copy()
    a["bodyId"] = a["bodyId"].astype(int)
    a["cluster"] = a["cluster"].astype(int)
    if "cluster_size" not in a.columns:
        sizes = a["cluster"].value_counts()
        a["cluster_size"] = a["cluster"].map(sizes).astype(int)
    if "cluster_name" not in a.columns:
        a["cluster_name"] = a["cluster"].map(lambda c: f"{dataset_key}_C{int(c):02d}")

    plot_df = emb.merge(a[["bodyId", "cluster", "cluster_size", "cluster_name"]], on="bodyId", how="inner")
    if len(plot_df) == 0:
        raise RuntimeError("Active UMAP and assignments have no overlapping bodyIds.")

    clusters = sorted(plot_df["cluster"].dropna().astype(int).unique())
    color_map = get_cluster_color_map(dataset_key, clusters)

    fig, ax = plt.subplots(figsize=(7.8, 6.5), dpi=FIG_DPI)
    for cl in clusters:
        sub = plot_df.loc[plot_df["cluster"].astype(int) == cl]
        ax.scatter(
            sub[xcol],
            sub[ycol],
            color=color_map[cl],
            s=42,
            alpha=0.92,
            edgecolor="white",
            linewidth=0.35,
            label=f"C{cl:02d} (n={len(sub)})",
        )

    if label_points and len(plot_df) <= 200:
        for _, r in plot_df.iterrows():
            ax.text(r[xcol], r[ycol], str(int(r["bodyId"])), fontsize=4.5, alpha=0.65)

    active_k = ds.get("active_k", plot_df["cluster"].nunique())
    active_linkage = ds.get("active_linkage_method", "")
    active_umap = ds.get("active_umap_name", "active")
    title = ds.get("label", dataset_key)
    if active_linkage or active_umap:
        title = f"Raphe superior soma-in-ROI | {active_linkage} k={active_k} | UMAP {active_umap}"

    ax.set_title(title)
    ax.set_xlabel("umap1")
    ax.set_ylabel("umap2")
    ax.set_aspect("equal", adjustable="datalim")
    if len(clusters) <= 16:
        ax.legend(frameon=False, fontsize=7, markerscale=0.9, loc="best")
    fig.tight_layout()

    out_base = FIG_DIR / f"active_umap_{dataset_key}_{active_linkage}_k{int(active_k):02d}_{active_umap}"
    savefig_all(fig, out_base)
    display(fig)
    plt.close(fig)

    print(f"Saved active UMAP: {out_base}")
    return plot_df


active_umap_plot_df = plot_active_umap_from_dataset(FIGURE_DATASET_KEY, label_points=False)

summary_rows = []
for key, ds in datasets.items():
    a = ds["assignments"]
    for cl, n in a.groupby("cluster").size().items():
        summary_rows.append({"dataset": key, "cluster": int(cl), "n_cells": int(n)})
summary_df = pd.DataFrame(summary_rows)
summary_path = PLOT_OUT_ROOT / "active_cluster_size_summary.csv"
summary_df.to_csv(summary_path, index=False)
print(f"Saved cluster-size summary: {summary_path}")
display(summary_df.sort_values(["dataset", "cluster"]))


In [ ]:
# ============================================================
# 4. Excel-friendly cluster ID exports from loaded assignments
# ============================================================

try:
    import openpyxl  # noqa
    HAS_OPENPYXL = True
except Exception:
    HAS_OPENPYXL = False


def export_loaded_cluster_ids(ds, dataset_key):
    a = ds["assignments"].copy().sort_values(["cluster", "bodyId"])
    a["dataset"] = dataset_key
    a["dataset_label"] = ds["label"]
    a["cluster_name"] = a["cluster"].map(lambda x: f"{dataset_key}_C{int(x):02d}")
    sizes = a["cluster"].value_counts()
    a["cluster_size"] = a["cluster"].map(sizes).astype(int)

    per_neuron_path = EXPORT_DIR / f"{dataset_key}_per_neuron_cluster_assignments_loaded.csv"
    a.to_csv(per_neuron_path, index=False)

    rows = []
    for cl, sub in a.groupby("cluster"):
        ids = sub["bodyId"].astype(int).tolist()
        rows.append({
            "dataset": dataset_key,
            "dataset_label": ds["label"],
            "cluster": int(cl),
            "cluster_name": f"{dataset_key}_C{int(cl):02d}",
            "n_cells": len(ids),
            "bodyIds_comma": ",".join(map(str, ids)),
            "bodyIds_semicolon": ";".join(map(str, ids)),
        })
    cluster_df = pd.DataFrame(rows).sort_values(["n_cells", "cluster"], ascending=[False, True])
    cluster_path = EXPORT_DIR / f"{dataset_key}_cluster_id_lists_loaded.csv"
    cluster_df.to_csv(cluster_path, index=False)

    print(f"Saved: {per_neuron_path}")
    print(f"Saved: {cluster_path}")

    if HAS_OPENPYXL:
        xlsx = EXPORT_DIR / f"{dataset_key}_cluster_ids_loaded.xlsx"
        with pd.ExcelWriter(xlsx, engine="openpyxl") as writer:
            a.to_excel(writer, sheet_name="per_neuron", index=False)
            cluster_df.to_excel(writer, sheet_name="per_cluster", index=False)
        print(f"Saved: {xlsx}")

    return cluster_df

all_cluster_summaries = []
for key, ds in datasets.items():
    all_cluster_summaries.append(export_loaded_cluster_ids(ds, key))

combined_cluster_summary = pd.concat(all_cluster_summaries, ignore_index=True)
combined_path = EXPORT_DIR / "combined_cluster_id_lists_loaded.csv"
combined_cluster_summary.to_csv(combined_path, index=False)
print(f"Saved: {combined_path}")
display(combined_cluster_summary)


In [ ]:
# ============================================================
# 5. SWC parser + display-only flipped coordinate transform
# ============================================================

SWC_COLS = ["node_id", "type", "x", "y", "z", "radius", "parent"]


def body_id_from_path(path):
    m = re.search(r"(\d+)", Path(path).stem)
    return int(m.group(1)) if m else None


def read_swc(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split()
            if len(parts) < 7:
                continue
            try:
                rows.append([
                    int(float(parts[0])), int(float(parts[1])),
                    float(parts[2]), float(parts[3]), float(parts[4]),
                    float(parts[5]), int(float(parts[6])),
                ])
            except Exception:
                pass
    return pd.DataFrame(rows, columns=SWC_COLS)


def swc_edges(df):
    if df is None or df.empty:
        return []
    ids = set(df["node_id"].astype(int))
    edges = []
    for _, r in df.iterrows():
        p = int(r["parent"])
        n = int(r["node_id"])
        if p in ids and p != -1:
            edges.append((p, n))
    return edges


def estimate_soma_from_swc(df):
    if df is None or df.empty:
        return None
    # Prefer SWC type 1 soma node if available.
    soma_nodes = df.loc[df["type"].astype(int) == 1]
    if len(soma_nodes):
        r = soma_nodes.iloc[0]
        return np.array([r["x"], r["y"], r["z"]], dtype=float)
    # Then root.
    root = df.loc[(df["parent"].astype(int) == -1)]
    if len(root):
        r = root.iloc[0]
        return np.array([r["x"], r["y"], r["z"]], dtype=float)
    # Then largest-radius node.
    if "radius" in df.columns and df["radius"].notna().any():
        r = df.loc[df["radius"].astype(float).idxmax()]
        return np.array([r["x"], r["y"], r["z"]], dtype=float)
    return df[["x", "y", "z"]].median().values.astype(float)


def swc_path_for_body(body_id, swc_dir):
    swc_dir = Path(swc_dir)
    direct = swc_dir / f"{int(body_id)}.swc"
    if direct.exists():
        return direct
    matches = list(swc_dir.glob(f"*{int(body_id)}*.swc"))
    return matches[0] if matches else None


def swc_segments_for_body(body_id, swc_dir):
    path = swc_path_for_body(body_id, swc_dir)
    if path is None:
        return [], None
    df = read_swc(path)
    if df.empty:
        return [], None
    coords = df.set_index("node_id")[["x", "y", "z"]]
    segs = []
    for p, n in swc_edges(df):
        try:
            a = coords.loc[p].values.astype(float)
            b = coords.loc[n].values.astype(float)
            segs.append((a, b))
        except Exception:
            pass
    soma = estimate_soma_from_swc(df)
    return segs, soma

# Load the shell from the previous v14 run.
SHELL_NPZ = PREVIOUS_OUT_ROOT / "outer_brain_shell" / "outer_brain_shell_voxel_union.npz"
if SHELL_NPZ.exists():
    shell_data = np.load(SHELL_NPZ)
    outer_shell_vertices = shell_data["vertices"]
    outer_shell_faces = shell_data["faces"].astype(int)
    print("Loaded outer shell:", SHELL_NPZ, outer_shell_vertices.shape, outer_shell_faces.shape)
else:
    outer_shell_vertices = None
    outer_shell_faces = None
    print("No previous outer shell found:", SHELL_NPZ)


# ------------------------------------------------------------
# Clean brain shell mesh: remove disconnected ROI/mask blobs
# ------------------------------------------------------------
# The raw shell built from all neuPrint ROI meshes can include small disconnected
# components outside the brain. For figure-ready rendering we keep only the
# largest connected mesh component, which gives a cleaner translucent brain
# outline like the reference figure.
SHELL_KEEP_LARGEST_COMPONENT_ONLY = True
SHELL_MIN_COMPONENT_FACE_FRACTION = 0.02


def clean_outer_shell_mesh_largest_component(vertices, faces, min_face_fraction=SHELL_MIN_COMPONENT_FACE_FRACTION):
    """
    Keep only the largest connected component of a triangular mesh.

    Connectivity is defined by faces sharing vertices. This removes small
    disconnected ROI blobs / masks outside the main brain shell.
    """
    if vertices is None or faces is None:
        return vertices, faces, {"status": "missing"}

    vertices = np.asarray(vertices, dtype=float)
    faces = np.asarray(faces, dtype=int)

    if len(vertices) == 0 or len(faces) == 0:
        return vertices, faces, {"status": "empty"}

    from collections import defaultdict, deque

    vertex_to_faces = defaultdict(list)
    for fi, tri in enumerate(faces):
        for v in tri:
            vertex_to_faces[int(v)].append(fi)

    seen = np.zeros(len(faces), dtype=bool)
    components = []

    for start in range(len(faces)):
        if seen[start]:
            continue
        q = deque([start])
        seen[start] = True
        comp = []
        while q:
            fi = q.popleft()
            comp.append(fi)
            for v in faces[fi]:
                for nb in vertex_to_faces[int(v)]:
                    if not seen[nb]:
                        seen[nb] = True
                        q.append(nb)
        components.append(np.asarray(comp, dtype=int))

    components = sorted(components, key=len, reverse=True)
    largest = components[0]
    min_faces = max(1, int(len(faces) * float(min_face_fraction)))

    # Default: largest component only. This is what removes the little blobs.
    keep_faces = largest

    kept_faces = faces[keep_faces]
    used_vertices = np.unique(kept_faces.ravel())
    old_to_new = {int(old): i for i, old in enumerate(used_vertices)}
    remapped_faces = np.vectorize(old_to_new.get)(kept_faces).astype(int)
    cleaned_vertices = vertices[used_vertices]

    info = {
        "status": "cleaned",
        "n_components": len(components),
        "original_vertices": int(len(vertices)),
        "original_faces": int(len(faces)),
        "kept_vertices": int(len(cleaned_vertices)),
        "kept_faces": int(len(remapped_faces)),
        "largest_component_faces": int(len(largest)),
        "min_faces_threshold": int(min_faces),
        "removed_faces": int(len(faces) - len(remapped_faces)),
        "component_face_counts_top10": [int(len(c)) for c in components[:10]],
    }

    print("Brain shell cleaning:")
    print(f"  components found: {info['n_components']}")
    print(f"  original faces: {info['original_faces']:,}")
    print(f"  kept faces:     {info['kept_faces']:,}")
    print(f"  removed faces:  {info['removed_faces']:,}")
    print(f"  top component face counts: {info['component_face_counts_top10']}")

    return cleaned_vertices, remapped_faces, info


if SHELL_KEEP_LARGEST_COMPONENT_ONLY and outer_shell_vertices is not None and outer_shell_faces is not None:
    outer_shell_vertices, outer_shell_faces, SHELL_CLEANING_INFO = clean_outer_shell_mesh_largest_component(
        outer_shell_vertices,
        outer_shell_faces,
    )
else:
    SHELL_CLEANING_INFO = {"status": "not_cleaned"}


def compute_render_center():
    if outer_shell_vertices is not None and len(outer_shell_vertices):
        return np.nanmean(outer_shell_vertices, axis=0)
    pts = []
    for ds in datasets.values():
        swc_dir = ds.get("raw_swc_dir")
        for bid in ds.get("body_ids", [])[:50]:
            path = swc_path_for_body(bid, swc_dir)
            if path:
                df = read_swc(path)
                if len(df):
                    pts.append(df[["x", "y", "z"]].values)
    if pts:
        return np.nanmean(np.vstack(pts), axis=0)
    return np.array([0.0, 0.0, 0.0])

RENDER_CENTER = compute_render_center()
print("Render center:", RENDER_CENTER)


def transform_points_for_render(points):
    """Apply display-only reflection around RENDER_CENTER."""
    if points is None:
        return None
    arr = np.asarray(points, dtype=float).copy()
    was_1d = arr.ndim == 1
    if was_1d:
        arr = arr[None, :]
    if arr.shape[1] < 3:
        return points
    center = np.asarray(RENDER_CENTER, dtype=float)
    if RENDER_FLIP_X:
        arr[:, 0] = 2 * center[0] - arr[:, 0]
    if RENDER_FLIP_Y:
        arr[:, 1] = 2 * center[1] - arr[:, 1]
    if RENDER_FLIP_Z:
        arr[:, 2] = 2 * center[2] - arr[:, 2]
    return arr[0] if was_1d else arr

outer_shell_vertices_render = transform_points_for_render(outer_shell_vertices) if outer_shell_vertices is not None else None
print("Render flips applied: X=", RENDER_FLIP_X, "Y=", RENDER_FLIP_Y, "Z=", RENDER_FLIP_Z)


In [ ]:
# ============================================================
# 6. neuPrint soma metadata: real somaLocation/somaRadius when available
# ============================================================

def get_neuprint_token():
    token = os.environ.get("NEUPRINT_TOKEN", "")
    if token:
        return token
    token_file = PROJECT_DIR / "neuprint_token.txt"
    if token_file.exists():
        token = token_file.read_text().strip()
        if token:
            os.environ["NEUPRINT_TOKEN"] = token
            return token
    if NEUPRINT_TOKEN_FALLBACK:
        os.environ["NEUPRINT_TOKEN"] = NEUPRINT_TOKEN_FALLBACK
        return NEUPRINT_TOKEN_FALLBACK
    return ""


def make_neuprint_client_or_none():
    if not HAS_NEUPRINT:
        return None
    token = get_neuprint_token()
    if not token:
        print("No neuPrint token found. Soma metadata will fall back to SWC estimates.")
        return None
    try:
        return Client(NEUPRINT_SERVER, dataset=NEUPRINT_DATASET, token=token)
    except Exception as e:
        print("Could not create neuPrint client. Soma metadata will fall back to SWC estimates.")
        print(repr(e))
        return None

c = make_neuprint_client_or_none()


def parse_neuprint_location_value(v):
    if v is None:
        return None
    try:
        if isinstance(v, float) and np.isnan(v):
            return None
    except Exception:
        pass
    if isinstance(v, str):
        nums = re.findall(r"[-+]?\d*\.\d+|[-+]?\d+", v)
        if len(nums) >= 3:
            return np.array([float(nums[0]), float(nums[1]), float(nums[2])], dtype=float)
        return None
    if isinstance(v, (list, tuple, np.ndarray)) and len(v) >= 3:
        return np.array([float(v[0]), float(v[1]), float(v[2])], dtype=float)
    if isinstance(v, dict):
        if all(k in v for k in ["x", "y", "z"]):
            return np.array([float(v["x"]), float(v["y"]), float(v["z"])], dtype=float)
        if "coordinates" in v and len(v["coordinates"]) >= 3:
            vv = v["coordinates"]
            return np.array([float(vv[0]), float(vv[1]), float(vv[2])], dtype=float)
    if all(hasattr(v, k) for k in ["x", "y", "z"]):
        return np.array([float(v.x), float(v.y), float(v.z)], dtype=float)
    return None


def fetch_neuprint_soma_metadata(body_ids, label="all_datasets"):
    body_ids = sorted(set(int(x) for x in body_ids if pd.notna(x)))
    if not body_ids or c is None or not USE_NEUPRINT_SOMAS:
        return pd.DataFrame()
    out_csv = SOMA_DIR / f"{label}_soma_metadata.csv"
    if out_csv.exists() and not FORCE_REFETCH_SOMAS:
        df = pd.read_csv(out_csv)
        print(f"Loaded cached soma metadata: {out_csv} | rows={len(df)}")
        return df
    rows = []
    batch_size = 300
    for i in tqdm(range(0, len(body_ids), batch_size), desc=f"Fetch neuPrint soma metadata {label}"):
        batch = body_ids[i:i+batch_size]
        try:
            neurons, _ = fetch_neurons(NC(bodyId=batch), client=c)
        except Exception as e:
            print(f"Soma metadata fetch failed for batch {i//batch_size}: {repr(e)}")
            continue
        if neurons is None or neurons.empty:
            continue
        for _, r in neurons.iterrows():
            bid = int(r["bodyId"])
            loc = None
            source = None
            for col in SOMA_LOCATION_PRIORITY:
                if col in neurons.columns:
                    loc = parse_neuprint_location_value(r.get(col, None))
                    if loc is not None:
                        source = col
                        break
            radius = np.nan
            for rad_col in ["somaRadius", "radius"]:
                if rad_col in neurons.columns:
                    try:
                        radius = float(r.get(rad_col))
                    except Exception:
                        radius = np.nan
                    break
            rows.append({
                "bodyId": bid,
                "soma_x": np.nan if loc is None else loc[0],
                "soma_y": np.nan if loc is None else loc[1],
                "soma_z": np.nan if loc is None else loc[2],
                "soma_radius": radius,
                "soma_source": "neuprint_" + str(source) if source else "neuprint_missing",
            })
    df = pd.DataFrame(rows).drop_duplicates("bodyId")
    df.to_csv(out_csv, index=False)
    print(f"Saved soma metadata: {out_csv} | rows={len(df)}")
    return df

all_body_ids_for_somas = []
for ds in datasets.values():
    all_body_ids_for_somas.extend(ds["assignments"]["bodyId"].astype(int).tolist())

soma_metadata_df = fetch_neuprint_soma_metadata(all_body_ids_for_somas, label="all_loaded_datasets")

SOMA_LOOKUP = {}
if not soma_metadata_df.empty:
    for _, r in soma_metadata_df.iterrows():
        loc = np.array([r.get("soma_x", np.nan), r.get("soma_y", np.nan), r.get("soma_z", np.nan)], dtype=float)
        if np.isfinite(loc).all():
            SOMA_LOOKUP[int(r["bodyId"])] = {
                "xyz": loc,
                "radius": float(r.get("soma_radius", np.nan)) if pd.notna(r.get("soma_radius", np.nan)) else np.nan,
                "source": r.get("soma_source", "neuprint"),
            }
print(f"Soma lookup entries with finite coordinates: {len(SOMA_LOOKUP)}")


## Soma/cell-body rendering note

This plotting version separates the visual inspection into three views per cluster:

1. **Skeletons only** — old morphology skeleton traces without soma meshes.
2. **Somas only** — local SWC-radius soma/root meshes without projections.
3. **Combined** — skeleton traces + soma/root meshes.

This should make it much easier to judge whether the soma rendering helps or distracts, and whether projection patterns are consistent within a cluster. All cells in a cluster use the same stable cluster color across all three views.

If true per-body segmentation meshes are available locally, put files in `body_meshes/<bodyId>.obj` or `body_meshes/<bodyId>.ply` and use `SOMA_RENDER_MODE = "local_body_mesh"`.

In [ ]:
# ============================================================
# 7. 3D rendering helpers: shell + neurons + SWC-radius soma meshes
# ============================================================

if not HAS_PLOTLY:
    raise ImportError("Plotly is required for the interactive 3D cluster renderings.")


def rgba(hex_color, alpha=1.0):
    """Convert '#RRGGBB' to plotly rgba string."""
    hex_color = str(hex_color).lstrip("#")
    r = int(hex_color[0:2], 16)
    g = int(hex_color[2:4], 16)
    b = int(hex_color[4:6], 16)
    return f"rgba({r},{g},{b},{alpha})"


def add_shell_trace(fig, alpha=SHELL_ALPHA):
    if not SHOW_BRAIN_SHELL:
        return fig
    if outer_shell_vertices_render is None or outer_shell_faces is None or len(outer_shell_vertices_render) == 0:
        return fig
    v = outer_shell_vertices_render
    f = outer_shell_faces
    fig.add_trace(go.Mesh3d(
        x=v[:, 0], y=v[:, 1], z=v[:, 2],
        i=f[:, 0], j=f[:, 1], k=f[:, 2],
        opacity=alpha,
        color="rgb(210, 215, 220)",
        name="outer brain shell",
        showscale=False,
        hoverinfo="skip",
        flatshading=False,
        lighting=dict(ambient=0.82, diffuse=0.35, specular=0.05, roughness=0.95),
    ))
    return fig


def soma_for_body(body_id, swc_dir):
    """Return soma center/radius/source using neuPrint metadata, then SWC fallback."""
    bid = int(body_id)
    if bid in SOMA_LOOKUP:
        d = SOMA_LOOKUP[bid]
        return d["xyz"], d.get("radius", np.nan), d.get("source", "neuprint")
    _, swc_soma = swc_segments_for_body(bid, swc_dir)
    if swc_soma is not None:
        return swc_soma, np.nan, "swc_fallback"
    return None, np.nan, "missing"


def sphere_mesh_vertices_faces(center, radius, theta_steps=9, phi_steps=7):
    """Return vertices/faces for a small sphere mesh."""
    center = np.asarray(center, dtype=float)
    radius = float(radius)
    theta = np.linspace(0, 2*np.pi, theta_steps, endpoint=False)
    phi = np.linspace(0, np.pi, phi_steps)
    verts = []
    for p in phi:
        for t in theta:
            verts.append([
                center[0] + radius * np.cos(t) * np.sin(p),
                center[1] + radius * np.sin(t) * np.sin(p),
                center[2] + radius * np.cos(p),
            ])
    faces = []
    for pi in range(phi_steps - 1):
        for ti in range(theta_steps):
            a = pi * theta_steps + ti
            b = pi * theta_steps + ((ti + 1) % theta_steps)
            c = (pi + 1) * theta_steps + ti
            d = (pi + 1) * theta_steps + ((ti + 1) % theta_steps)
            faces.append([a, c, b])
            faces.append([b, c, d])
    return np.asarray(verts, dtype=float), np.asarray(faces, dtype=int)


def cylinder_mesh_vertices_faces(a, b, radius, sides=9):
    """Return vertices/faces for a cylinder between two 3D points."""
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    axis = b - a
    length = np.linalg.norm(axis)
    if not np.isfinite(length) or length <= 1e-9:
        return np.empty((0, 3)), np.empty((0, 3), dtype=int)
    axis = axis / length
    # Choose an arbitrary perpendicular basis.
    tmp = np.array([1.0, 0.0, 0.0])
    if abs(np.dot(tmp, axis)) > 0.9:
        tmp = np.array([0.0, 1.0, 0.0])
    u = np.cross(axis, tmp)
    u = u / np.linalg.norm(u)
    v = np.cross(axis, u)
    angles = np.linspace(0, 2*np.pi, sides, endpoint=False)
    ring_a, ring_b = [], []
    for t in angles:
        off = radius * (np.cos(t) * u + np.sin(t) * v)
        ring_a.append(a + off)
        ring_b.append(b + off)
    verts = np.vstack([ring_a, ring_b])
    faces = []
    for i in range(sides):
        j = (i + 1) % sides
        faces.append([i, j, sides + i])
        faces.append([j, sides + j, sides + i])
    return verts, np.asarray(faces, dtype=int)


def combine_mesh_parts(parts):
    """Combine list of (vertices, faces) mesh parts."""
    verts_all = []
    faces_all = []
    offset = 0
    for verts, faces in parts:
        if verts is None or faces is None or len(verts) == 0 or len(faces) == 0:
            continue
        verts_all.append(verts)
        faces_all.append(faces + offset)
        offset += len(verts)
    if not verts_all:
        return np.empty((0, 3)), np.empty((0, 3), dtype=int)
    return np.vstack(verts_all), np.vstack(faces_all)


def add_mesh3d(fig, vertices, faces, color, name, opacity=0.9, hoverinfo="skip", showlegend=False):
    if vertices is None or faces is None or len(vertices) == 0 or len(faces) == 0:
        return fig
    v = transform_points_for_render(vertices)
    fig.add_trace(go.Mesh3d(
        x=v[:, 0], y=v[:, 1], z=v[:, 2],
        i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
        color=color,
        opacity=opacity,
        name=name,
        showscale=False,
        hoverinfo=hoverinfo,
        flatshading=False,
        lighting=dict(ambient=0.45, diffuse=0.65, specular=0.2, roughness=0.75),
        showlegend=showlegend,
    ))
    return fig


def closest_swc_node_to_point(df, point):
    if df is None or df.empty or point is None:
        return None
    pts = df[["x", "y", "z"]].values.astype(float)
    point = np.asarray(point, dtype=float)
    d = np.linalg.norm(pts - point[None, :], axis=1)
    return int(df.iloc[int(np.nanargmin(d))]["node_id"])


def soma_anchor_node(df, soma_xyz=None):
    """Best node to anchor soma rendering."""
    if df is None or df.empty:
        return None
    if soma_xyz is not None and np.isfinite(np.asarray(soma_xyz, dtype=float)).all():
        n = closest_swc_node_to_point(df, soma_xyz)
        if n is not None:
            return n
    soma_nodes = df.loc[df["type"].astype(int) == 1]
    if len(soma_nodes):
        return int(soma_nodes.iloc[0]["node_id"])
    roots = df.loc[df["parent"].astype(int) == -1]
    if len(roots):
        return int(roots.iloc[0]["node_id"])
    if "radius" in df.columns and df["radius"].notna().any():
        return int(df.loc[df["radius"].astype(float).idxmax()]["node_id"])
    return int(df.iloc[0]["node_id"])


def local_soma_node_set(df, soma_xyz, anchor):
    """Choose compact local SWC node set around soma/root, emphasizing large radii."""
    df = df.copy()
    ids = set(df["node_id"].astype(int))
    coords = df.set_index("node_id")[["x", "y", "z"]]
    radii = df.set_index("node_id")["radius"].astype(float)
    adjacency = defaultdict(list)
    for p, n in swc_edges(df):
        if p in ids and n in ids:
            adjacency[int(p)].append(int(n))
            adjacency[int(n)].append(int(p))

    visited = {int(anchor): 0}
    frontier = [int(anchor)]
    for _ in range(SOMA_RADIUS_MESH_GRAPH_HOPS):
        new_frontier = []
        for u in frontier:
            for v in adjacency.get(u, []):
                if v not in visited:
                    visited[v] = visited[u] + 1
                    new_frontier.append(v)
        frontier = new_frontier
        if not frontier:
            break
    graph_nodes = set(visited.keys())

    if soma_xyz is None or not np.isfinite(np.asarray(soma_xyz, dtype=float)).all():
        soma_xyz = coords.loc[anchor].values.astype(float)
    soma_xyz = np.asarray(soma_xyz, dtype=float)
    all_pts = coords.values.astype(float)
    all_ids = coords.index.astype(int).to_numpy()
    d = np.linalg.norm(all_pts - soma_xyz[None, :], axis=1)
    euclidean_nodes = set(all_ids[d <= SOMA_RADIUS_MESH_EUCLIDEAN_RADIUS].astype(int))

    # Include local high-radius nodes; this tends to capture soma/root swelling better than distance alone.
    rr = radii.reindex(all_ids).values.astype(float)
    valid_rr = rr[np.isfinite(rr) & (rr > 0)]
    high_radius_nodes = set()
    if len(valid_rr):
        threshold = np.nanpercentile(valid_rr, 85)
        high_radius_nodes = set(all_ids[(rr >= threshold) & (d <= SOMA_RADIUS_MESH_EUCLIDEAN_RADIUS * 1.4)].astype(int))

    chosen = list((graph_nodes | euclidean_nodes | high_radius_nodes).intersection(ids))
    if len(chosen) < SOMA_RADIUS_MESH_MIN_NODES:
        nearest = all_ids[np.argsort(d)[:SOMA_RADIUS_MESH_MIN_NODES]].astype(int).tolist()
        chosen = list(set(chosen) | set(nearest))
    if len(chosen) > SOMA_RADIUS_MESH_MAX_NODES:
        chosen_arr = np.array(chosen, dtype=int)
        pts_chosen = coords.loc[chosen_arr].values.astype(float)
        dd = np.linalg.norm(pts_chosen - soma_xyz[None, :], axis=1)
        # Prefer nearby + high radius.
        rr_chosen = radii.loc[chosen_arr].values.astype(float)
        rr_norm = np.nan_to_num(rr_chosen / (np.nanmax(rr_chosen) if np.nanmax(rr_chosen) > 0 else 1), nan=0)
        score = dd - 0.5 * SOMA_RADIUS_MESH_EUCLIDEAN_RADIUS * rr_norm
        chosen = chosen_arr[np.argsort(score)[:SOMA_RADIUS_MESH_MAX_NODES]].astype(int).tolist()
    return sorted(set(map(int, chosen)))


def build_swc_radius_soma_mesh(body_id, swc_dir):
    """Build a compact soma/root mesh from local SWC nodes/radii."""
    path = swc_path_for_body(body_id, swc_dir)
    if path is None:
        return None
    df = read_swc(path)
    if df.empty:
        return None

    soma_xyz, soma_radius, soma_source = soma_for_body(body_id, swc_dir)
    anchor = soma_anchor_node(df, soma_xyz)
    if anchor is None:
        return None

    coords = df.set_index("node_id")[["x", "y", "z"]]
    radii = df.set_index("node_id")["radius"].astype(float)
    chosen = local_soma_node_set(df, soma_xyz, anchor)
    chosen_set = set(chosen)

    # Radii: use SWC radius, with conservative clipping to avoid ugly giant blobs.
    rr = radii.loc[chosen].values.astype(float)
    good = np.isfinite(rr) & (rr > 0)
    fallback_r = np.nanmedian(rr[good]) if np.any(good) else (soma_radius if np.isfinite(soma_radius) and soma_radius > 0 else 60.0)
    rr = np.where(good, rr, fallback_r)
    rr = np.clip(rr * SOMA_RADIUS_MESH_RADIUS_SCALE, SOMA_RADIUS_MESH_MIN_RADIUS, SOMA_RADIUS_MESH_MAX_RADIUS)

    parts = []
    # Sphere caps at selected local nodes.
    for nid, radius in zip(chosen, rr):
        center = coords.loc[nid].values.astype(float)
        parts.append(sphere_mesh_vertices_faces(
            center, radius,
            theta_steps=SOMA_RADIUS_MESH_SPHERE_THETA,
            phi_steps=SOMA_RADIUS_MESH_SPHERE_PHI,
        ))

    # Cylinders along local SWC edges. Radius is the smaller of endpoint radii.
    rr_by_id = dict(zip(chosen, rr))
    for p, n in swc_edges(df):
        if p in chosen_set and n in chosen_set:
            a = coords.loc[p].values.astype(float)
            b = coords.loc[n].values.astype(float)
            rad = min(rr_by_id.get(p, fallback_r), rr_by_id.get(n, fallback_r))
            parts.append(cylinder_mesh_vertices_faces(a, b, rad, sides=SOMA_RADIUS_MESH_CYLINDER_SIDES))

    vertices, faces = combine_mesh_parts(parts)
    if len(vertices) == 0 or len(faces) == 0:
        return None
    return {
        "vertices": vertices,
        "faces": faces,
        "source": soma_source,
        "n_nodes": len(chosen),
        "center": soma_xyz if soma_xyz is not None else coords.loc[anchor].values.astype(float),
    }


def read_obj_mesh(path):
    verts, faces = [], []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if line.startswith("v "):
                parts = line.split()
                if len(parts) >= 4:
                    verts.append([float(parts[1]), float(parts[2]), float(parts[3])])
            elif line.startswith("f "):
                inds = []
                for p in line.split()[1:]:
                    try:
                        inds.append(int(p.split("/")[0]) - 1)
                    except Exception:
                        pass
                if len(inds) >= 3:
                    # Triangulate fan if needed.
                    for i in range(1, len(inds)-1):
                        faces.append([inds[0], inds[i], inds[i+1]])
    return np.asarray(verts, dtype=float), np.asarray(faces, dtype=int)


def read_ply_ascii_mesh(path):
    """Minimal ASCII PLY reader. Binary PLY is not supported here."""
    with open(path, "r", errors="ignore") as f:
        lines = f.readlines()
    if not lines or not lines[0].startswith("ply"):
        raise ValueError("Not a PLY file")
    n_vertices = n_faces = None
    end_idx = None
    for i, line in enumerate(lines):
        if line.startswith("format") and "ascii" not in line:
            raise ValueError("Only ASCII PLY supported by this lightweight reader")
        if line.startswith("element vertex"):
            n_vertices = int(line.split()[-1])
        if line.startswith("element face"):
            n_faces = int(line.split()[-1])
        if line.strip() == "end_header":
            end_idx = i + 1
            break
    if end_idx is None or n_vertices is None:
        raise ValueError("Malformed PLY")
    verts = []
    for line in lines[end_idx:end_idx+n_vertices]:
        parts = line.split()
        verts.append([float(parts[0]), float(parts[1]), float(parts[2])])
    faces = []
    face_start = end_idx + n_vertices
    for line in lines[face_start:face_start+(n_faces or 0)]:
        vals = [int(x) for x in line.split()]
        k = vals[0]
        inds = vals[1:1+k]
        for i in range(1, len(inds)-1):
            faces.append([inds[0], inds[i], inds[i+1]])
    return np.asarray(verts, dtype=float), np.asarray(faces, dtype=int)


def local_body_mesh_path(body_id):
    if not LOCAL_BODY_MESH_DIR.exists():
        return None
    for ext in [".obj", ".ply"]:
        p = LOCAL_BODY_MESH_DIR / f"{int(body_id)}{ext}"
        if p.exists():
            return p
    matches = []
    for ext in ["*.obj", "*.ply"]:
        matches.extend(LOCAL_BODY_MESH_DIR.glob(f"*{int(body_id)}*{ext[-4:]}"))
    return matches[0] if matches else None


def clipped_local_body_soma_mesh(body_id, swc_dir):
    """If local true body mesh exists, clip soma neighborhood and return mesh."""
    path = local_body_mesh_path(body_id)
    if path is None:
        return None
    soma_xyz, soma_radius, source = soma_for_body(body_id, swc_dir)
    if soma_xyz is None:
        return None
    try:
        if str(path).lower().endswith(".obj"):
            verts, faces = read_obj_mesh(path)
        else:
            verts, faces = read_ply_ascii_mesh(path)
    except Exception as e:
        print(f"Could not read local body mesh for {body_id}: {e}")
        return None
    if len(verts) == 0 or len(faces) == 0:
        return None
    d = np.linalg.norm(verts - np.asarray(soma_xyz)[None, :], axis=1)
    keep_v = d <= LOCAL_BODY_MESH_SOMA_CLIP_RADIUS
    keep_faces = keep_v[faces].all(axis=1)
    faces2 = faces[keep_faces]
    if len(faces2) == 0:
        return None
    # Limit if too large.
    if len(faces2) > LOCAL_BODY_MESH_MAX_FACES:
        rng = np.random.default_rng(42)
        faces2 = faces2[np.sort(rng.choice(len(faces2), LOCAL_BODY_MESH_MAX_FACES, replace=False))]
    used = np.unique(faces2.ravel())
    remap = {old: new for new, old in enumerate(used)}
    verts_new = verts[used]
    faces_new = np.vectorize(remap.get)(faces2)
    return {"vertices": verts_new, "faces": faces_new.astype(int), "source": f"local_body_mesh:{path.name}"}


def add_soma_trace(fig, body_id, swc_dir, color, show_text=False):
    if SOMA_RENDER_MODE == "local_body_mesh" or (PREFER_LOCAL_BODY_MESH_SOMA and SOMA_RENDER_MODE == "swc_radius_mesh"):
        mesh = clipped_local_body_soma_mesh(body_id, swc_dir)
        if mesh is not None:
            fig = add_mesh3d(
                fig, mesh["vertices"], mesh["faces"],
                color=color,
                name=f"true/local soma mesh {int(body_id)}",
                opacity=LOCAL_BODY_MESH_OPACITY,
                hoverinfo="text",
                showlegend=False,
            )
            return fig, True

    if SOMA_RENDER_MODE == "swc_radius_mesh":
        mesh = build_swc_radius_soma_mesh(body_id, swc_dir)
        if mesh is not None:
            fig = add_mesh3d(
                fig, mesh["vertices"], mesh["faces"],
                color=color,
                name=f"SWC-radius soma mesh {int(body_id)}",
                opacity=SOMA_RADIUS_MESH_OPACITY,
                hoverinfo="skip",
                showlegend=False,
            )
            # Optional label at mesh center.
            if show_text:
                c = transform_points_for_render(mesh["center"])
                fig.add_trace(go.Scatter3d(
                    x=[c[0]], y=[c[1]], z=[c[2]], mode="text",
                    text=[str(int(body_id))], textposition="top center",
                    showlegend=False, hoverinfo="skip",
                ))
            return fig, str(mesh.get("source", "")).startswith("neuprint")

    # Simple marker/sphere fallback.
    soma, radius, source = soma_for_body(body_id, swc_dir)
    if soma is None:
        return fig, False
    soma_r = transform_points_for_render(soma)
    label = f"soma {int(body_id)} ({source})"
    if SOMA_RENDER_MODE == "sphere" and np.isfinite(radius) and radius > 0:
        verts, faces = sphere_mesh_vertices_faces(soma, radius * SOMA_SPHERE_RADIUS_SCALE, theta_steps=10, phi_steps=8)
        fig = add_mesh3d(fig, verts, faces, color=color, name=label, opacity=SOMA_SPHERE_OPACITY)
    else:
        size = SOMA_MARKER_SIZE
        if np.isfinite(radius) and radius > 0:
            size = int(np.clip(radius / 50, SOMA_MARKER_SIZE_MIN, SOMA_MARKER_SIZE_MAX))
        fig.add_trace(go.Scatter3d(
            x=[soma_r[0]], y=[soma_r[1]], z=[soma_r[2]],
            mode="markers+text" if show_text else "markers",
            marker=dict(size=size, symbol="circle", color=color),
            text=[str(int(body_id))] if show_text else None,
            textposition="top center",
            name=label,
            hovertext=label,
            hoverinfo="text",
            showlegend=False,
        ))
    return fig, source.startswith("neuprint")


def build_neuron_radius_tube_mesh(body_id, swc_dir):
    """Experimental full-skeleton tube mesh from SWC radii. Heavy but tunable."""
    path = swc_path_for_body(body_id, swc_dir)
    if path is None:
        return None
    df = read_swc(path)
    if df.empty:
        return None
    coords = df.set_index("node_id")[["x", "y", "z"]]
    radii = df.set_index("node_id")["radius"].astype(float)
    parts = []
    edges = swc_edges(df)[:CELL_TUBE_MAX_EDGES_PER_NEURON]
    for p, n in edges:
        try:
            a = coords.loc[p].values.astype(float)
            b = coords.loc[n].values.astype(float)
            r = np.nanmean([radii.loc[p], radii.loc[n]])
            if not np.isfinite(r) or r <= 0:
                r = CELL_TUBE_MIN_RADIUS
            r = np.clip(r * CELL_TUBE_RADIUS_SCALE, CELL_TUBE_MIN_RADIUS, CELL_TUBE_MAX_RADIUS)
            parts.append(cylinder_mesh_vertices_faces(a, b, r, sides=CELL_TUBE_SIDES))
        except Exception:
            pass
    verts, faces = combine_mesh_parts(parts)
    if len(verts) == 0:
        return None
    return {"vertices": verts, "faces": faces}


def add_single_neuron_trace(fig, body_id, swc_dir, cluster_color, width=CELL_LINE_WIDTH):
    if CELL_RENDER_MODE == "radius_tubes":
        mesh = build_neuron_radius_tube_mesh(body_id, swc_dir)
        if mesh is not None:
            return add_mesh3d(
                fig, mesh["vertices"], mesh["faces"],
                color=rgba(cluster_color, CELL_LINE_ALPHA),
                name=f"neuron radius tube {int(body_id)}",
                opacity=CELL_LINE_ALPHA,
                hoverinfo="skip",
                showlegend=False,
            )
    # Fast clean line rendering fallback/default.
    segs, _ = swc_segments_for_body(body_id, swc_dir)
    if not segs:
        return fig
    xs, ys, zs = [], [], []
    for a, b in segs:
        aa = transform_points_for_render(a)
        bb = transform_points_for_render(b)
        xs += [aa[0], bb[0], None]
        ys += [aa[1], bb[1], None]
        zs += [aa[2], bb[2], None]
    fig.add_trace(go.Scatter3d(
        x=xs, y=ys, z=zs,
        mode="lines",
        line=dict(width=width, color=rgba(cluster_color, CELL_LINE_ALPHA)),
        name=f"cluster neuron {int(body_id)}",
        hovertext=str(int(body_id)),
        hoverinfo="text",
        showlegend=False,
    ))
    return fig


def add_neuron_traces(fig, body_ids, swc_dir, cluster_color, width=CELL_LINE_WIDTH, show_somas=True, show_soma_text=False):
    n_soma_neuprint = 0
    for bid in body_ids:
        fig = add_single_neuron_trace(fig, bid, swc_dir, cluster_color=cluster_color, width=width)
        if show_somas:
            fig, is_neuprint = add_soma_trace(fig, bid, swc_dir, color=cluster_color, show_text=show_soma_text)
            n_soma_neuprint += int(is_neuprint)
    return fig, n_soma_neuprint


def add_population_context(fig, ds, max_cells=MAX_CONTEXT_CELLS):
    a = ds["assignments"]
    ids = a["bodyId"].astype(int).tolist()[:max_cells]
    swc_dir = ds["raw_swc_dir"]
    for bid in ids:
        segs, _ = swc_segments_for_body(bid, swc_dir)
        if not segs:
            continue
        xs, ys, zs = [], [], []
        for p, q in segs:
            pp = transform_points_for_render(p)
            qq = transform_points_for_render(q)
            xs += [pp[0], qq[0], None]
            ys += [pp[1], qq[1], None]
            zs += [pp[2], qq[2], None]
        fig.add_trace(go.Scatter3d(
            x=xs, y=ys, z=zs,
            mode="lines",
            line=dict(width=CONTEXT_LINE_WIDTH, color=f"rgba(0,0,0,{POPULATION_CONTEXT_ALPHA})"),
            showlegend=False,
            hoverinfo="skip",
        ))
    return fig


def render_cluster_3d(ds, dataset_key, cluster, save_html=True, show_population_context=False, show_somas=True, show_soma_text=False):
    a = ds["assignments"]
    cluster = int(cluster)
    body_ids = a.loc[a["cluster"] == cluster, "bodyId"].astype(int).tolist()
    swc_dir = ds["raw_swc_dir"]
    ccolor = get_cluster_color(dataset_key, cluster)

    fig = go.Figure()
    add_shell_trace(fig, alpha=SHELL_ALPHA)
    if show_population_context:
        add_population_context(fig, ds)
    body_ids_to_draw = body_ids[:MAX_CELLS_PER_CLUSTER_RENDER]
    fig, n_neuprint_somas = add_neuron_traces(
        fig,
        body_ids_to_draw,
        swc_dir,
        cluster_color=ccolor,
        width=CELL_LINE_WIDTH,
        show_somas=show_somas,
        show_soma_text=show_soma_text,
    )

    id_text = ", ".join(map(str, body_ids[:35])) + ("..." if len(body_ids) > 35 else "")
    title = (
        f"{ds['label']} | cluster {cluster} | n={len(body_ids)}"
        f"<br>cluster color: {ccolor} | skeleton mode: {CELL_RENDER_MODE} | line width: {CELL_LINE_WIDTH} | soma mode: {SOMA_RENDER_MODE}"
        f"<br>neuPrint soma centers used: {n_neuprint_somas}/{len(body_ids_to_draw)} | IDs: {id_text}"
    )
    fig.update_layout(
        title=title,
        scene=dict(
            xaxis_title="x", yaxis_title="y", zaxis_title="z",
            aspectmode="data",
            xaxis=dict(showbackground=False),
            yaxis=dict(showbackground=False),
            zaxis=dict(showbackground=False),
        ),
        width=1050,
        height=850,
        legend=dict(itemsizing="constant"),
        margin=dict(l=0, r=0, t=110, b=0),
    )
    if save_html and SAVE_INTERACTIVE_HTML:
        outdir = RENDER_DIR / dataset_key / "interactive_3d_v17_radius_soma_meshes"
        outdir.mkdir(parents=True, exist_ok=True)
        html = outdir / f"{dataset_key}_cluster_{cluster:02d}_v17_radiusSomaMesh.html"
        fig.write_html(str(html))
        print(f"Saved HTML: {html}")
    return fig


def inspect_cluster(dataset_key="selected_raphe", cluster=1, show_population_context=False, show_somas=True, show_soma_text=False):
    ds = datasets[dataset_key]
    ids = ds["assignments"].loc[ds["assignments"]["cluster"] == int(cluster), "bodyId"].astype(int).tolist()
    print(f"{ds['label']} | cluster {int(cluster)} | n={len(ids)} | color={get_cluster_color(dataset_key, cluster)}")
    print("IDs:", ", ".join(map(str, ids)))
    display(pd.DataFrame({"bodyId": ids}))
    fig = render_cluster_3d(
        ds, dataset_key, int(cluster),
        save_html=True,
        show_population_context=show_population_context,
        show_somas=show_somas,
        show_soma_text=show_soma_text,
    )
    display(fig)
    return fig


# ============================================================
# v18 overrides: render skeleton-only, soma-only, or combined views
# ============================================================

RENDER_MODE_LABELS = {
    "skeletons_only": "Skeletons only",
    "somas_only": "Somas only",
    "combined": "Skeletons + somas",
}


def _safe_cluster_ids(ds, cluster):
    a = ds["assignments"]
    cluster = int(cluster)
    return a.loc[a["cluster"] == cluster, "bodyId"].astype(int).tolist()


def render_cluster_3d_mode(
    ds,
    dataset_key,
    cluster,
    render_mode="combined",
    save_html=True,
    show_population_context=False,
    show_soma_text=False,
):
    """
    Render one cluster in one of three modes:
      - skeletons_only: only the old skeleton traces
      - somas_only: only soma/root radius meshes
      - combined: skeleton traces + soma/root meshes
    """
    if render_mode not in RENDER_MODE_LABELS:
        raise ValueError(f"render_mode must be one of {list(RENDER_MODE_LABELS)}")

    cluster = int(cluster)
    body_ids = _safe_cluster_ids(ds, cluster)
    body_ids_to_draw = body_ids[:MAX_CELLS_PER_CLUSTER_RENDER]
    swc_dir = ds["raw_swc_dir"]
    ccolor = get_cluster_color(dataset_key, cluster)

    fig = go.Figure()
    add_shell_trace(fig, alpha=SHELL_ALPHA)

    # Context is usually most useful in skeleton/combined mode; allow it everywhere if requested.
    if show_population_context:
        add_population_context(fig, ds)

    n_somas = 0
    for bid in body_ids_to_draw:
        if render_mode in ["skeletons_only", "combined"]:
            fig = add_single_neuron_trace(
                fig,
                bid,
                swc_dir,
                cluster_color=ccolor,
                width=CELL_LINE_WIDTH,
            )
        if render_mode in ["somas_only", "combined"]:
            fig, is_neuprint = add_soma_trace(
                fig,
                bid,
                swc_dir,
                color=ccolor,
                show_text=show_soma_text,
            )
            n_somas += int(is_neuprint)

    id_text = ", ".join(map(str, body_ids[:35])) + ("..." if len(body_ids) > 35 else "")
    title = (
        f"{ds['label']} | cluster {cluster} | n={len(body_ids)} | {RENDER_MODE_LABELS[render_mode]}"
        f"<br>cluster color: {ccolor} | line width: {CELL_LINE_WIDTH} | soma mode: {SOMA_RENDER_MODE}"
        f"<br>somas drawn: {len(body_ids_to_draw) if render_mode in ['somas_only', 'combined'] else 0}; "
        f"neuPrint soma anchors: {n_somas}/{len(body_ids_to_draw)} | IDs: {id_text}"
    )

    fig.update_layout(
        title=title,
        scene=dict(
            xaxis_title="x", yaxis_title="y", zaxis_title="z",
            aspectmode="data",
            xaxis=dict(showbackground=False),
            yaxis=dict(showbackground=False),
            zaxis=dict(showbackground=False),
        ),
        width=1050,
        height=850,
        legend=dict(itemsizing="constant"),
        margin=dict(l=0, r=0, t=120, b=0),
    )

    if save_html and SAVE_INTERACTIVE_HTML and SAVE_EACH_RENDER_MODE_HTML:
        outdir = RENDER_DIR / dataset_key / "interactive_3d_v18_three_brain_views"
        outdir.mkdir(parents=True, exist_ok=True)
        html = outdir / f"{dataset_key}_cluster_{cluster:02d}_{render_mode}_v18.html"
        fig.write_html(str(html))
        print(f"Saved HTML: {html}")

    return fig


def render_cluster_three_views(
    ds,
    dataset_key,
    cluster,
    save_html=True,
    show_population_context=False,
    show_soma_text=False,
):
    """Render the three requested views as three separate interactive Plotly brains."""
    figs = {}
    for mode in ["skeletons_only", "somas_only", "combined"]:
        print(f"Rendering {RENDER_MODE_LABELS[mode]}...")
        figs[mode] = render_cluster_3d_mode(
            ds,
            dataset_key,
            cluster,
            render_mode=mode,
            save_html=save_html,
            show_population_context=show_population_context,
            show_soma_text=show_soma_text,
        )
    return figs


# Backward-compatible wrapper. Existing calls still work, but default to combined.
def render_cluster_3d(
    ds,
    dataset_key,
    cluster,
    save_html=True,
    show_population_context=False,
    show_somas=True,
    show_soma_text=False,
    render_mode="combined",
):
    if not show_somas and render_mode == "combined":
        render_mode = "skeletons_only"
    return render_cluster_3d_mode(
        ds,
        dataset_key,
        cluster,
        render_mode=render_mode,
        save_html=save_html,
        show_population_context=show_population_context,
        show_soma_text=show_soma_text,
    )


def inspect_cluster(
    dataset_key="selected_raphe",
    cluster=1,
    show_population_context=False,
    show_somas=True,
    show_soma_text=False,
    render_mode=DEFAULT_CLUSTER_RENDER_MODE,
):
    """
    Inspect a cluster. By default, render_mode='all_three' shows:
    skeletons-only, somas-only, and combined.
    """
    ds = datasets[dataset_key]
    ids = _safe_cluster_ids(ds, cluster)
    print(f"{ds['label']} | cluster {int(cluster)} | n={len(ids)} | color={get_cluster_color(dataset_key, cluster)}")
    print("IDs:", ", ".join(map(str, ids)))
    display(pd.DataFrame({"bodyId": ids}))

    if render_mode == "all_three":
        figs = render_cluster_three_views(
            ds,
            dataset_key,
            int(cluster),
            save_html=True,
            show_population_context=show_population_context,
            show_soma_text=show_soma_text,
        )
        for mode, fig in figs.items():
            display(HTML(f"<h3>{RENDER_MODE_LABELS[mode]}</h3>"))
            display(fig)
        return figs

    fig = render_cluster_3d_mode(
        ds,
        dataset_key,
        int(cluster),
        render_mode=render_mode,
        save_html=True,
        show_population_context=show_population_context,
        show_soma_text=show_soma_text,
    )
    display(fig)
    return fig


In [ ]:
# ============================================================
# 8b. Plot soma/root locations of the neurons used, inside clean shell
# ============================================================
# Self-contained version: no dependency on later cells like save_plotly_all_formats,
# and no dependency on undefined helpers like find_swc_path.

SOMA_LOCATION_DATASET_KEY = globals().get("FIGURE_DATASET_KEY", "raphe_superior_soma_in_roi")

if "FIGURE_READY_DIR" not in globals():
    FIGURE_READY_DIR = PLOT_OUT_ROOT / "figure_ready_reclustered_fixed_3d_shell_soma_in_roi"
    FIGURE_READY_DIR.mkdir(parents=True, exist_ok=True)

SOMA_LOCATION_FIG_DIR = FIGURE_READY_DIR / "soma_locations_in_clean_shell"
SOMA_LOCATION_FIG_DIR.mkdir(parents=True, exist_ok=True)

SOMA_LOCATION_COLOR = "rgb(184, 134, 11)"  # muted golden/yellow
SOMA_LOCATION_SIZE = 3.0
SOMA_LOCATION_ALPHA = 0.80
SOMA_SHELL_ALPHA = 0.030
SOMA_LOCATION_WIDTH = 1200
SOMA_LOCATION_HEIGHT = 1500
SOMA_LOCATION_SCALE = 2


def save_plotly_figure_selfcontained(fig, out_base, scale=SOMA_LOCATION_SCALE, save_html=True):
    """Save a Plotly figure without relying on later notebook helper cells."""
    out_base = Path(out_base)
    out_base.parent.mkdir(parents=True, exist_ok=True)

    if save_html and globals().get("SAVE_INTERACTIVE_HTML", True):
        html_path = out_base.with_suffix(".html")
        fig.write_html(str(html_path))
        print("Saved HTML:", html_path)

    try:
        if globals().get("SAVE_PNG", True):
            png_path = out_base.with_suffix(".png")
            fig.write_image(str(png_path), scale=scale)
            print("Saved PNG:", png_path)
        if globals().get("SAVE_PDF", True):
            pdf_path = out_base.with_suffix(".pdf")
            fig.write_image(str(pdf_path), scale=scale)
            print("Saved PDF:", pdf_path)
        if globals().get("SAVE_SVG", True):
            svg_path = out_base.with_suffix(".svg")
            fig.write_image(str(svg_path), scale=scale)
            print("Saved SVG:", svg_path)
    except Exception as e:
        print(
            "Static export failed, but HTML was saved if enabled.\n"
            "To enable static export, install/update kaleido:\n"
            "  pip install -U kaleido\n"
            f"Error: {repr(e)}"
        )


def find_swc_path_for_soma_location(body_id, ds=None):
    """
    Robust SWC path lookup for this plot-only notebook.
    Uses the dataset's raw_swc_dir first, then common fallback folders.
    """
    body_id = int(body_id)
    candidate_dirs = []

    if ds is not None:
        for key in ["raw_swc_dir", "swc_dir", "skeleton_dir"]:
            if key in ds and ds[key] is not None:
                candidate_dirs.append(Path(ds[key]))

    # Canonical and fallback directories used in the notebooks.
    candidate_dirs.extend([
        RAPHE_SUPERIOR_SOMA_IN_ROI_RAW_SWC_DIR,
        SKELETON_DIR / "raphe_superior_soma_in_roi_raw_swc",
        SKELETON_DIR / "raphe_superior_raw_swc",
        PLOT_OUT_ROOT / "skeletons" / "raphe_superior_soma_in_roi_raw_swc",
        PLOT_OUT_ROOT / "raphe_superior_soma_in_roi_raw_swc",
        PREVIOUS_OUT_ROOT / "skeletons" / "raphe_superior_soma_in_roi_raw_swc",
        PREVIOUS_OUT_ROOT / "raphe_superior_soma_in_roi_raw_swc",
    ])

    seen_dirs = set()
    for d in candidate_dirs:
        if d is None:
            continue
        d = Path(d)
        try:
            key = str(d.resolve())
        except Exception:
            key = str(d)
        if key in seen_dirs:
            continue
        seen_dirs.add(key)

        if not d.exists():
            continue

        direct = d / f"{body_id}.swc"
        if direct.exists():
            return direct

        matches = sorted(d.glob(f"*{body_id}*.swc"))
        if matches:
            return matches[0]

    # Last limited fallback.
    if SKELETON_DIR.exists():
        matches = sorted(SKELETON_DIR.glob(f"**/*{body_id}*.swc"))
        if matches:
            return matches[0]

    return None


def read_swc_for_soma_location(path):
    """Use notebook read_swc() if available; otherwise parse SWC here."""
    if path is None:
        return pd.DataFrame(columns=["node_id", "type", "x", "y", "z", "radius", "parent"])
    if "read_swc" in globals():
        return read_swc(path)

    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split()
            if len(parts) < 7:
                continue
            try:
                rows.append([
                    int(float(parts[0])),
                    int(float(parts[1])),
                    float(parts[2]),
                    float(parts[3]),
                    float(parts[4]),
                    float(parts[5]),
                    int(float(parts[6])),
                ])
            except Exception:
                continue
    return pd.DataFrame(rows, columns=["node_id", "type", "x", "y", "z", "radius", "parent"])


def estimate_soma_root_xyz_for_body(body_id, ds=None):
    if ds is None:
        ds = datasets[SOMA_LOCATION_DATASET_KEY]

    swc_path = find_swc_path_for_soma_location(body_id, ds)
    if swc_path is None:
        return None, None

    df = read_swc_for_soma_location(swc_path)
    if df is None or df.empty:
        return None, swc_path

    # Priority: explicit soma node > largest-radius node > root node > median coordinate.
    try:
        soma = df.loc[df["type"].astype(int) == 1]
        if len(soma):
            r = soma.iloc[0]
            return np.array([r["x"], r["y"], r["z"]], dtype=float), swc_path
    except Exception:
        pass

    try:
        if "radius" in df.columns and df["radius"].notna().any():
            r = df.loc[df["radius"].astype(float).idxmax()]
            return np.array([r["x"], r["y"], r["z"]], dtype=float), swc_path
    except Exception:
        pass

    try:
        root = df.loc[df["parent"].astype(int) < 0]
        if len(root):
            r = root.iloc[0]
            return np.array([r["x"], r["y"], r["z"]], dtype=float), swc_path
    except Exception:
        pass

    return df[["x", "y", "z"]].median().values.astype(float), swc_path


def collect_active_soma_locations(dataset_key=SOMA_LOCATION_DATASET_KEY):
    ds = datasets[dataset_key]
    body_ids = ds["assignments"]["bodyId"].astype(int).unique().tolist()

    rows = []
    missing = []
    for bid in body_ids:
        xyz, path = estimate_soma_root_xyz_for_body(bid, ds)
        if xyz is None or not np.isfinite(xyz).all():
            missing.append(int(bid))
            continue
        rows.append({
            "bodyId": int(bid),
            "x": float(xyz[0]),
            "y": float(xyz[1]),
            "z": float(xyz[2]),
            "swc_path": "" if path is None else str(path),
        })

    out = pd.DataFrame(rows)
    print(f"{dataset_key}: soma/root coordinates found {len(out)} / {len(body_ids)}")
    if missing:
        print("First missing:", missing[:20])

    out_path = SOMA_LOCATION_FIG_DIR / f"{dataset_key}_active_soma_root_locations.csv"
    out.to_csv(out_path, index=False)
    print("Saved:", out_path)
    return out


def _fixed_soma_camera(view):
    view = str(view).lower()
    if view == "top":
        return dict(eye=dict(x=0.0, y=0.0, z=2.35), up=dict(x=0, y=1, z=0))
    if view == "side":
        return dict(eye=dict(x=2.35, y=0.0, z=0.0), up=dict(x=0, y=0, z=1))
    if view == "angled":
        return dict(eye=dict(x=1.55, y=1.65, z=1.15), up=dict(x=0, y=0, z=1))
    raise ValueError(view)


def plot_active_soma_locations_in_shell(dataset_key=SOMA_LOCATION_DATASET_KEY, views=("top", "side", "angled")):
    if not HAS_PLOTLY:
        raise RuntimeError("Plotly required")
    if "add_shell_trace" not in globals():
        raise RuntimeError("add_shell_trace() is missing. Run the 3D shell helper cell first.")

    soma_df = collect_active_soma_locations(dataset_key)
    if soma_df.empty:
        raise RuntimeError("No soma/root coordinates found to plot.")

    pts = soma_df[["x", "y", "z"]].to_numpy(float)
    pts_render = transform_points_for_render(pts)
    soma_df["x_render"] = pts_render[:, 0]
    soma_df["y_render"] = pts_render[:, 1]
    soma_df["z_render"] = pts_render[:, 2]

    figs = {}
    for view in views:
        fig = go.Figure()
        add_shell_trace(fig, alpha=SOMA_SHELL_ALPHA)

        fig.add_trace(go.Scatter3d(
            x=soma_df["x_render"],
            y=soma_df["y_render"],
            z=soma_df["z_render"],
            mode="markers",
            marker=dict(
                size=SOMA_LOCATION_SIZE,
                color=SOMA_LOCATION_COLOR,
                opacity=SOMA_LOCATION_ALPHA,
            ),
            text=[f"bodyId: {int(b)}" for b in soma_df["bodyId"]],
            hoverinfo="text",
            name=f"Soma/root locations, n={len(soma_df)}",
        ))

        fig.update_layout(
            title=dict(
                text=f"{datasets[dataset_key]['label']}<br>Soma/root locations | {view}",
                x=0.5,
                xanchor="center",
            ),
            scene=dict(
                xaxis=dict(visible=False, showbackground=False),
                yaxis=dict(visible=False, showbackground=False),
                zaxis=dict(visible=False, showbackground=False),
                aspectmode="data",
                camera=_fixed_soma_camera(view),
                bgcolor="white",
            ),
            paper_bgcolor="white",
            plot_bgcolor="white",
            width=SOMA_LOCATION_WIDTH,
            height=SOMA_LOCATION_HEIGHT,
            margin=dict(l=0, r=0, t=70, b=0),
            showlegend=False,
        )

        display(fig)
        out_base = SOMA_LOCATION_FIG_DIR / f"{dataset_key}_soma_locations_{view}"
        save_plotly_figure_selfcontained(fig, out_base, scale=SOMA_LOCATION_SCALE, save_html=True)
        figs[view] = fig

    return figs, soma_df


soma_location_figs, soma_location_table = plot_active_soma_locations_in_shell()


In [ ]:
# ============================================================
# 8c. Soma/root locations colored by active cluster ID
# ============================================================
# This uses the same clean brain shell and the same fixed-camera helper as the
# other static 3D exports. The "side" view is a true left/right lateral view.

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path
from IPython.display import display

CLUSTER_SOMA_FIG_DIR = FIG_DIR / "soma_locations_colored_by_cluster_in_clean_shell"
CLUSTER_SOMA_FIG_DIR.mkdir(parents=True, exist_ok=True)

CLUSTER_SOMA_DATASET_KEY = FIGURE_DATASET_KEY
CLUSTER_SOMA_SIZE = 4.2
CLUSTER_SOMA_ALPHA = 0.92
CLUSTER_SOMA_SHOW_LEGEND = True
CLUSTER_SOMA_SHOW_TEXT_LABELS = False
CLUSTER_SOMA_SHELL_ALPHA = 0.045
CLUSTER_SOMA_SHELL_COLOR = "rgb(185, 190, 195)"
CLUSTER_SOMA_STATIC_WIDTH = 1500
CLUSTER_SOMA_STATIC_HEIGHT = 1200
CLUSTER_SOMA_STATIC_SCALE = 2
SAVE_CLUSTER_SOMA_STATIC = True
SAVE_CLUSTER_SOMA_HTML = True

# "side" uses the central true-lateral y-axis camera.
# To save both lateral views, use ["top", "left", "right", "angled"].
CLUSTER_SOMA_VIEWS = ["top", "side", "angled"]


def _require_cluster_soma_inputs():
    required = [
        "datasets", "FIGURE_DATASET_KEY", "FIG_DIR",
        "outer_shell_vertices_render", "outer_shell_faces",
        "transform_points_for_render", "camera_for_static_3d_view",
    ]
    missing = [x for x in required if x not in globals()]
    if missing:
        raise RuntimeError(
            "Missing required variables/functions: "
            + ", ".join(missing)
            + ". Run earlier setup/shell cells first."
        )
    if CLUSTER_SOMA_DATASET_KEY not in datasets:
        raise RuntimeError(
            f"Dataset {CLUSTER_SOMA_DATASET_KEY!r} not found. "
            f"Available datasets: {list(datasets.keys())}"
        )


def _get_soma_location_table_for_cluster_coloring():
    if "soma_location_table" in globals() and isinstance(soma_location_table, pd.DataFrame):
        tab = soma_location_table.copy()
        print(f"Using existing soma_location_table: {len(tab):,} rows")
    elif "collect_active_soma_locations" in globals():
        tab = collect_active_soma_locations(CLUSTER_SOMA_DATASET_KEY).copy()
        print(f"Generated soma table with collect_active_soma_locations(): {len(tab):,} rows")
    else:
        raise RuntimeError(
            "Could not find soma_location_table and collect_active_soma_locations() is not defined. "
            "Run the previous soma-location 3D cell first."
        )

    if "bodyId" not in tab.columns:
        raise RuntimeError("soma_location_table must contain a bodyId column.")

    needed_raw = {"x", "y", "z"}
    needed_render = {"x_render", "y_render", "z_render"}
    if not needed_raw.issubset(tab.columns) and not needed_render.issubset(tab.columns):
        raise RuntimeError(
            "soma_location_table must contain either x/y/z or x_render/y_render/z_render columns."
        )

    tab["bodyId"] = tab["bodyId"].astype(int)
    return tab


def _get_active_assignment_table_for_cluster_coloring():
    ds = datasets[CLUSTER_SOMA_DATASET_KEY]
    if "assignments" not in ds:
        raise RuntimeError(f"datasets[{CLUSTER_SOMA_DATASET_KEY!r}] has no assignments table.")
    a = ds["assignments"].copy()
    if "bodyId" not in a.columns or "cluster" not in a.columns:
        raise RuntimeError("Active assignments must contain bodyId and cluster columns.")
    a["bodyId"] = a["bodyId"].astype(int)
    a["cluster"] = a["cluster"].astype(int)
    return a


def _cluster_color_map_for_soma_points(assignments):
    clusters = sorted(assignments["cluster"].dropna().astype(int).unique())
    if "get_cluster_color" in globals():
        return {cl: get_cluster_color(CLUSTER_SOMA_DATASET_KEY, cl) for cl in clusters}
    palette = [
        "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd",
        "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf",
        "#aec7e8", "#ffbb78", "#98df8a", "#ff9896", "#c5b0d5",
        "#c49c94", "#f7b6d2", "#c7c7c7", "#dbdb8d", "#9edae5",
        "#393b79", "#637939", "#8c6d31", "#843c39", "#7b4173",
        "#3182bd", "#31a354", "#756bb1", "#636363", "#e6550d",
    ]
    return {cl: palette[(cl - 1) % len(palette)] for cl in clusters}


def _mesh3d_trace_cluster_soma(vertices, faces, name, color, opacity):
    vertices = np.asarray(vertices, dtype=float)
    faces = np.asarray(faces, dtype=int)
    return go.Mesh3d(
        x=vertices[:, 0], y=vertices[:, 1], z=vertices[:, 2],
        i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
        name=name,
        color=color,
        opacity=opacity,
        flatshading=True,
        lighting=dict(ambient=0.72, diffuse=0.55, specular=0.08, roughness=0.88, fresnel=0.03),
        lightposition=dict(x=0, y=0, z=100000),
        hoverinfo="name",
        showscale=False,
    )


def save_cluster_soma_plotly_figure(fig, out_base):
    if SAVE_CLUSTER_SOMA_HTML:
        html_path = out_base.with_suffix(".html")
        fig.write_html(str(html_path))
        print(f"Saved HTML: {html_path}")
    if SAVE_CLUSTER_SOMA_STATIC:
        try:
            if globals().get("SAVE_PNG", True):
                png_path = out_base.with_suffix(".png")
                fig.write_image(str(png_path), scale=CLUSTER_SOMA_STATIC_SCALE)
                print(f"Saved PNG: {png_path}")
            if globals().get("SAVE_PDF", True):
                pdf_path = out_base.with_suffix(".pdf")
                fig.write_image(str(pdf_path), scale=CLUSTER_SOMA_STATIC_SCALE)
                print(f"Saved PDF: {pdf_path}")
            if globals().get("SAVE_SVG", True):
                svg_path = out_base.with_suffix(".svg")
                fig.write_image(str(svg_path), scale=CLUSTER_SOMA_STATIC_SCALE)
                print(f"Saved SVG: {svg_path}")
        except Exception as e:
            print(
                "Static export failed, but HTML was saved if enabled.\n"
                "To enable static export, install/update kaleido:\n"
                "  pip install -U kaleido\n"
                f"Error: {repr(e)}"
            )


def plot_soma_locations_colored_by_cluster_in_shell():
    _require_cluster_soma_inputs()
    soma_df = _get_soma_location_table_for_cluster_coloring()
    assignments = _get_active_assignment_table_for_cluster_coloring()

    plot_df = soma_df.merge(assignments[["bodyId", "cluster"]], on="bodyId", how="inner")
    missing_from_assignments = sorted(set(soma_df["bodyId"].astype(int)) - set(plot_df["bodyId"].astype(int)))
    if missing_from_assignments:
        print(f"Warning: {len(missing_from_assignments):,} soma points missing from active assignments.")
        print("First missing:", missing_from_assignments[:20])
    if len(plot_df) == 0:
        raise RuntimeError("No soma/root points matched active assignments.")

    if {"x_render", "y_render", "z_render"}.issubset(plot_df.columns):
        plot_df["xr"] = plot_df["x_render"].astype(float)
        plot_df["yr"] = plot_df["y_render"].astype(float)
        plot_df["zr"] = plot_df["z_render"].astype(float)
    else:
        raw = plot_df[["x", "y", "z"]].to_numpy(float)
        rr = transform_points_for_render(raw)
        plot_df["xr"] = rr[:, 0]
        plot_df["yr"] = rr[:, 1]
        plot_df["zr"] = rr[:, 2]

    cluster_sizes = plot_df["cluster"].value_counts().sort_index()
    color_map = _cluster_color_map_for_soma_points(assignments)

    print("Cluster-colored soma/root points:")
    print(f"  plotted cells: {len(plot_df):,}")
    print(f"  clusters: {plot_df['cluster'].nunique():,}")
    display(cluster_sizes.rename("n_soma_points").reset_index().rename(columns={"index": "cluster"}))

    table_path = CLUSTER_SOMA_FIG_DIR / "soma_root_locations_colored_by_active_cluster.csv"
    plot_df.to_csv(table_path, index=False)
    print(f"Saved cluster-colored soma/root table: {table_path}")

    brain_vertices = np.asarray(outer_shell_vertices_render, dtype=float)
    brain_faces = np.asarray(outer_shell_faces, dtype=int)
    figs = {}

    for view in CLUSTER_SOMA_VIEWS:
        fig = go.Figure()
        fig.add_trace(
            _mesh3d_trace_cluster_soma(
                brain_vertices,
                brain_faces,
                name="Clean outer brain shell",
                color=CLUSTER_SOMA_SHELL_COLOR,
                opacity=CLUSTER_SOMA_SHELL_ALPHA,
            )
        )

        for cl in sorted(plot_df["cluster"].astype(int).unique()):
            sub = plot_df.loc[plot_df["cluster"].astype(int) == cl]
            color = color_map[int(cl)]
            hover_text = [
                f"bodyId: {int(bid)}<br>cluster: C{int(cl):02d}<br>x={x:.1f}<br>y={y:.1f}<br>z={z:.1f}"
                for bid, x, y, z in zip(sub["bodyId"], sub["x"], sub["y"], sub["z"])
            ]
            mode = "markers+text" if CLUSTER_SOMA_SHOW_TEXT_LABELS else "markers"
            fig.add_trace(
                go.Scatter3d(
                    x=sub["xr"], y=sub["yr"], z=sub["zr"],
                    mode=mode,
                    marker=dict(
                        size=CLUSTER_SOMA_SIZE,
                        color=color,
                        opacity=CLUSTER_SOMA_ALPHA,
                        line=dict(width=0.4, color="rgba(0,0,0,0.35)"),
                    ),
                    text=[str(int(x)) for x in sub["bodyId"]],
                    textposition="top center",
                    name=f"C{int(cl):02d} (n={len(sub)})",
                    hovertext=hover_text,
                    hoverinfo="text",
                    showlegend=CLUSTER_SOMA_SHOW_LEGEND,
                )
            )

        active_label = (
            f"{globals().get('ACTIVE_LINKAGE_METHOD', 'active')}, "
            f"k={globals().get('ACTIVE_CLUSTER_K', 'active')}, "
            f"UMAP={globals().get('ACTIVE_UMAP_NAME', 'active')}"
        )
        fig.update_layout(
            title=dict(
                text=f"Soma/root locations colored by active cluster — {view}<br><sup>{active_label}</sup>",
                x=0.5,
                xanchor="center",
            ),
            scene=dict(
                xaxis=dict(visible=False),
                yaxis=dict(visible=False),
                zaxis=dict(visible=False),
                aspectmode="data",
                camera=camera_for_static_3d_view(view),
                bgcolor="white",
            ),
            paper_bgcolor="white",
            plot_bgcolor="white",
            margin=dict(l=0, r=0, t=72, b=0),
            width=CLUSTER_SOMA_STATIC_WIDTH,
            height=CLUSTER_SOMA_STATIC_HEIGHT,
            showlegend=CLUSTER_SOMA_SHOW_LEGEND,
            legend=dict(x=0.02, y=0.98, bgcolor="rgba(255,255,255,0.72)", font=dict(size=10)),
        )

        display(fig)
        out_base = CLUSTER_SOMA_FIG_DIR / f"soma_root_locations_by_active_cluster_{view}"
        save_cluster_soma_plotly_figure(fig, out_base)
        figs[view] = fig

    return figs, plot_df


cluster_colored_soma_figs, cluster_colored_soma_table = plot_soma_locations_colored_by_cluster_in_shell()


In [ ]:
# ============================================================
# 8. Interactive cluster selector: three separate brain views
# ============================================================

def launch_cluster_selector():
    if not HAS_WIDGETS:
        print("ipywidgets not available. Use inspect_cluster('selected_raphe', 1, render_mode='all_three').")
        return

    dataset_options = [(ds["label"], key) for key, ds in datasets.items()]
    dataset_dd = widgets.Dropdown(options=dataset_options, description="Dataset:", layout=widgets.Layout(width="420px"))
    cluster_dd = widgets.Dropdown(description="Cluster:", layout=widgets.Layout(width="260px"))
    render_mode_dd = widgets.Dropdown(
        options=[
            ("All three: skeletons, somas, combined", "all_three"),
            ("Skeletons only", "skeletons_only"),
            ("Somas only", "somas_only"),
            ("Combined", "combined"),
        ],
        value=DEFAULT_CLUSTER_RENDER_MODE,
        description="View:",
        layout=widgets.Layout(width="420px"),
    )
    show_context = widgets.Checkbox(value=False, description="Population context")
    show_soma_text = widgets.Checkbox(value=False, description="Soma ID labels")
    out = widgets.Output()

    def update_clusters(*args):
        key = dataset_dd.value
        ds = datasets[key]
        sizes = ds["assignments"]["cluster"].value_counts().sort_index()
        opts = [(f"C{int(c):02d} | n={int(s)}", int(c)) for c, s in sizes.items()]
        cluster_dd.options = opts

    def draw(*args):
        with out:
            clear_output(wait=True)
            key = dataset_dd.value
            cl = cluster_dd.value
            if cl is None:
                return
            inspect_cluster(
                key,
                cl,
                show_population_context=show_context.value,
                show_soma_text=show_soma_text.value,
                render_mode=render_mode_dd.value,
            )

    dataset_dd.observe(update_clusters, names="value")
    cluster_dd.observe(draw, names="value")
    render_mode_dd.observe(draw, names="value")
    show_context.observe(draw, names="value")
    show_soma_text.observe(draw, names="value")

    update_clusters()
    display(widgets.VBox([
        widgets.HTML("<b>Interactive cluster renderer — choose skeletons-only, somas-only, combined, or all three</b>"),
        widgets.HBox([dataset_dd, cluster_dd]),
        widgets.HBox([render_mode_dd]),
        widgets.HBox([show_context, show_soma_text]),
        out,
    ]))
    draw()

launch_cluster_selector()


In [ ]:
# ============================================================
# 9. Optional: batch-generate HTML renders for all non-singleton clusters
# ============================================================

# Set to True and rerun this cell if you want to regenerate HTML for every non-singleton cluster.
BATCH_RENDER_ALL_NON_SINGLETON_CLUSTERS = False

# Choose which views to batch-save.
# Options: ["skeletons_only", "somas_only", "combined"]
BATCH_RENDER_MODES = ["skeletons_only", "somas_only", "combined"]

if BATCH_RENDER_ALL_NON_SINGLETON_CLUSTERS:
    for key, ds in datasets.items():
        sizes = ds["assignments"]["cluster"].value_counts().sort_values(ascending=False)
        clusters = [int(c) for c, s in sizes.items() if int(s) >= 2]
        print(f"{key}: rendering {len(clusters)} non-singleton clusters")
        for cl in tqdm(clusters, desc=f"Render {key}"):
            for mode in BATCH_RENDER_MODES:
                _ = render_cluster_3d_mode(
                    ds,
                    key,
                    cl,
                    render_mode=mode,
                    save_html=True,
                    show_population_context=False,
                    show_soma_text=False,
                )
else:
    print("Batch rendering disabled. Set BATCH_RENDER_ALL_NON_SINGLETON_CLUSTERS = True to save all cluster HTML renders.")


In [ ]:
# ============================================================
# 10. Check CELLS_TO_CHECK in active clusters + UMAP + brain plot
# ============================================================
# Uses the active reclustering chosen in Cell 2b.

CHECK_IDS_DIR = FIG_DIR / "checked_ids_active_recluster"
CHECK_IDS_DIR.mkdir(parents=True, exist_ok=True)

CHECK_ID_COLOR = "rgb(255, 220, 0)"  # yellow
CHECK_ID_LINE_COLOR = "#FFD700"
CHECK_ID_LINE_WIDTH = 3.2
CHECK_ID_SOMA_COLOR = "#B8860B"
CHECK_ID_SOMA_TEXT = True
CHECK_ID_SHELL_ALPHA = 0.030
CHECK_ID_STATIC_WIDTH = 1350
CHECK_ID_STATIC_HEIGHT = 1100
CHECK_ID_STATIC_SCALE = 2


def get_active_embedding_df(dataset_key=FIGURE_DATASET_KEY, method="umap"):
    """Robustly return the active embedding DataFrame, regardless of old/new storage format."""
    ds = datasets[dataset_key]
    emb_obj = ds.get("embeddings", None)

    if isinstance(emb_obj, dict):
        emb = emb_obj.get(method)
    elif isinstance(emb_obj, pd.DataFrame):
        # Old notebooks stored the UMAP directly as a DataFrame.
        emb = emb_obj if method == "umap" else None
    else:
        emb = None

    if emb is None and method == "umap" and "active_embedding_df" in ds:
        emb = ds["active_embedding_df"]

    if emb is None:
        raise RuntimeError(f"No active {method} embedding found for {dataset_key}. Run the reclustering cell first.")

    emb = emb.copy()
    emb["bodyId"] = emb["bodyId"].astype(int)
    return emb


def cluster_body_ids(ds, cluster):
    """Return body IDs in a cluster from a dataset assignment table."""
    a = ds["assignments"].copy()
    a["bodyId"] = a["bodyId"].astype(int)
    a["cluster"] = a["cluster"].astype(int)
    return a.loc[a["cluster"] == int(cluster), "bodyId"].astype(int).tolist()

# Alias for older rendering helpers that use _safe_cluster_ids.
_safe_cluster_ids = cluster_body_ids


def check_cells_in_active_clusters(cell_ids=None, dataset_key=FIGURE_DATASET_KEY):
    if cell_ids is None:
        cell_ids = CELLS_TO_CHECK
    cell_ids = [int(x) for x in cell_ids]

    ds = datasets[dataset_key]
    a = ds["assignments"].copy()
    a["bodyId"] = a["bodyId"].astype(int)
    a["cluster"] = a["cluster"].astype(int)

    rows = []
    for bid in cell_ids:
        hit = a.loc[a["bodyId"] == bid]
        if len(hit):
            for _, r in hit.iterrows():
                cl = int(r["cluster"])
                ids = cluster_body_ids(ds, cl)
                rows.append({
                    "bodyId": bid,
                    "dataset": dataset_key,
                    "dataset_label": ds.get("label", dataset_key),
                    "cluster": cl,
                    "cluster_name": f"{dataset_key}_C{cl:02d}",
                    "cluster_size": len(ids),
                    "active_linkage_method": ds.get("active_linkage_method", ""),
                    "active_k": ds.get("active_k", ""),
                    "active_umap_name": ds.get("active_umap_name", ""),
                    "cluster_bodyIds_semicolon": ";".join(map(str, ids)),
                    "status": "found_in_active_soma_in_roi_clustering",
                })
        else:
            rows.append({
                "bodyId": bid,
                "dataset": dataset_key,
                "dataset_label": ds.get("label", dataset_key),
                "cluster": np.nan,
                "cluster_name": "",
                "cluster_size": np.nan,
                "active_linkage_method": ds.get("active_linkage_method", ""),
                "active_k": ds.get("active_k", ""),
                "active_umap_name": ds.get("active_umap_name", ""),
                "cluster_bodyIds_semicolon": "",
                "status": "not_found_in_active_QC_passing_soma_in_roi_clustering",
            })

    out = pd.DataFrame(rows)
    display(out)
    path = CHECK_IDS_DIR / "checked_ids_cluster_membership_active_soma_in_roi.csv"
    out.to_csv(path, index=False)
    print("Saved:", path)
    return out


def plot_checked_ids_on_active_umap(cell_ids=None, dataset_key=FIGURE_DATASET_KEY):
    if cell_ids is None:
        cell_ids = CELLS_TO_CHECK
    cell_ids = sorted(set(int(x) for x in cell_ids))

    ds = datasets[dataset_key]
    a = ds["assignments"].copy()
    a["bodyId"] = a["bodyId"].astype(int)
    a["cluster"] = a["cluster"].astype(int)
    emb = get_active_embedding_df(dataset_key, "umap")
    df = emb.merge(a[["bodyId", "cluster", "cluster_size", "cluster_name"]], on="bodyId", how="inner")

    fig, ax = plt.subplots(figsize=(7.2, 6.2), dpi=360)
    for cl, sub in df.groupby("cluster", sort=True):
        cl = int(cl)
        ax.scatter(
            sub["umap1"], sub["umap2"],
            s=35,
            color=get_cluster_color(dataset_key, cl),
            alpha=0.88,
            edgecolor="white",
            linewidth=0.35,
            label=f"C{cl:02d} (n={len(sub)})",
        )

    h = df.loc[df["bodyId"].isin(cell_ids)].copy()
    if len(h):
        ax.scatter(
            h["umap1"], h["umap2"],
            s=190,
            marker="*",
            color="yellow",
            edgecolor="black",
            linewidth=0.85,
            zorder=10,
            label="CELLS_TO_CHECK",
        )
        for _, r in h.iterrows():
            ax.text(r["umap1"], r["umap2"], str(int(r["bodyId"])), fontsize=6, color="black", zorder=11)

    missing = sorted(set(cell_ids) - set(h["bodyId"].astype(int).tolist()))
    if missing:
        print("IDs not found in active UMAP/QC-passing set:", missing)

    ax.set_title(f"{ds.get('label', dataset_key)} | checked IDs highlighted")
    ax.set_xlabel("umap1")
    ax.set_ylabel("umap2")
    ax.set_aspect("equal", adjustable="datalim")
    if df["cluster"].nunique() <= 18:
        ax.legend(frameon=False, fontsize=6.5, loc="best", markerscale=0.9)
    fig.tight_layout()

    out_base = CHECK_IDS_DIR / "checked_ids_on_active_umap"
    savefig_all(fig, out_base)
    display(fig)
    plt.close(fig)
    return fig, df, h


def _save_plotly_checked_id_fig(fig, out_base):
    out_base = Path(out_base)
    out_base.parent.mkdir(parents=True, exist_ok=True)
    if SAVE_INTERACTIVE_HTML:
        p = out_base.with_suffix(".html")
        fig.write_html(str(p))
        print("Saved HTML:", p)
    try:
        if SAVE_PNG:
            p = out_base.with_suffix(".png")
            fig.write_image(str(p), scale=CHECK_ID_STATIC_SCALE)
            print("Saved PNG:", p)
        if SAVE_PDF:
            p = out_base.with_suffix(".pdf")
            fig.write_image(str(p), scale=CHECK_ID_STATIC_SCALE)
            print("Saved PDF:", p)
        if SAVE_SVG:
            p = out_base.with_suffix(".svg")
            fig.write_image(str(p), scale=CHECK_ID_STATIC_SCALE)
            print("Saved SVG:", p)
    except Exception as e:
        print("Static Plotly export failed; HTML/display is still usable. Error:", repr(e))


def _camera_for_checked_id_view(view):
    view = str(view).lower()
    if view == "top":
        return dict(eye=dict(x=0.0, y=0.0, z=2.35), up=dict(x=0, y=1, z=0))
    if view == "side":
        return dict(eye=dict(x=2.35, y=0.0, z=0.0), up=dict(x=0, y=0, z=1))
    if view == "angled":
        return dict(eye=dict(x=1.55, y=1.65, z=1.15), up=dict(x=0, y=0, z=1))
    raise ValueError(view)


def plot_checked_ids_in_brain_shell(cell_ids=None, dataset_key=FIGURE_DATASET_KEY, views=("top", "side", "angled")):
    """Plot checked-ID skeletons/somas in the clean brain shell."""
    if cell_ids is None:
        cell_ids = CELLS_TO_CHECK
    cell_ids = [int(x) for x in cell_ids]

    ds = datasets[dataset_key]
    a = ds["assignments"].copy()
    a["bodyId"] = a["bodyId"].astype(int)
    found_ids = [bid for bid in cell_ids if bid in set(a["bodyId"])]
    missing_ids = sorted(set(cell_ids) - set(found_ids))
    print(f"Checked IDs found in active QC-passing dataset: {len(found_ids)} / {len(cell_ids)}")
    if missing_ids:
        print("Missing/not clustered:", missing_ids)
    if not found_ids:
        raise RuntimeError("None of CELLS_TO_CHECK are present in the active clustered dataset.")

    figs = {}
    for view in views:
        fig = go.Figure()
        add_shell_trace(fig, alpha=CHECK_ID_SHELL_ALPHA)
        # Draw skeletons and somas in yellow/gold regardless of cluster.
        fig, n_somas = add_neuron_traces(
            fig,
            found_ids,
            ds["raw_swc_dir"],
            cluster_color=CHECK_ID_LINE_COLOR,
            width=CHECK_ID_LINE_WIDTH,
            show_somas=True,
            show_soma_text=CHECK_ID_SOMA_TEXT,
        )
        title = f"CELLS_TO_CHECK in brain shell | n={len(found_ids)} | {view} view"
        fig.update_layout(
            title=dict(text=title, x=0.5, xanchor="center"),
            scene=dict(
                xaxis=dict(visible=False, showbackground=False),
                yaxis=dict(visible=False, showbackground=False),
                zaxis=dict(visible=False, showbackground=False),
                aspectmode="data",
                camera=_camera_for_checked_id_view(view),
                bgcolor="white",
            ),
            paper_bgcolor="white",
            plot_bgcolor="white",
            margin=dict(l=0, r=0, t=60, b=0),
            width=CHECK_ID_STATIC_WIDTH,
            height=CHECK_ID_STATIC_HEIGHT,
            showlegend=False,
        )
        display(fig)
        out_base = CHECK_IDS_DIR / f"checked_ids_in_brain_shell_{view}"
        _save_plotly_checked_id_fig(fig, out_base)
        figs[view] = fig

    # Save found/missing list.
    pd.DataFrame({"bodyId": found_ids}).to_csv(CHECK_IDS_DIR / "checked_ids_found_in_active_dataset.csv", index=False)
    pd.DataFrame({"bodyId": missing_ids}).to_csv(CHECK_IDS_DIR / "checked_ids_missing_from_active_dataset.csv", index=False)
    return figs, found_ids, missing_ids


checked_cells_active = check_cells_in_active_clusters()
checked_ids_umap_fig, checked_ids_umap_df, checked_ids_umap_hits = plot_checked_ids_on_active_umap()
checked_ids_brain_figs, checked_ids_found_in_brain, checked_ids_missing_from_brain = plot_checked_ids_in_brain_shell()


## 11. Figure-ready exports — fixed 3D shell views

This section replaces the older 2D projection panels. The old panels used a projected/convex-hull brain outline, which looked ugly and did **not** match the nice interactive 3D brain.

The functions below reuse the same Plotly 3D rendering machinery as the interactive cluster viewer, including the same translucent outer brain shell mesh. The only difference is that the camera is fixed to top, side, and angled views and each view is exported as PDF/PNG/SVG.


In [ ]:

# ============================================================
# 11. Figure-ready export configuration — SOMA-IN-RAPHE-SUPERIOR
# ============================================================

FIGURE_READY_DIR = PLOT_OUT_ROOT / "figure_ready_reclustered_fixed_3d_shell_soma_in_roi"
FIGURE_READY_DIR.mkdir(parents=True, exist_ok=True)

FIG_UMAP_DIR = FIGURE_READY_DIR / "umap_mds"
FIG_CLUSTER_3D_DIR = FIGURE_READY_DIR / "cluster_3d_static_views"
FIG_SINGLE_CLUSTER_DIR = FIGURE_READY_DIR / "single_cluster_custom_3d"
for _p in [FIG_UMAP_DIR, FIG_CLUSTER_3D_DIR, FIG_SINGLE_CLUSTER_DIR]:
    _p.mkdir(parents=True, exist_ok=True)

# This notebook/version is for the Raphe-superior soma-in-ROI dataset.
FIGURE_DATASET_KEY = "raphe_superior_soma_in_roi"

# 3D static camera views.
FIGURE_3D_VIEWS = ["top", "side", "angled"]
FIGURE_3D_VIEW_LABELS = {
    "top": "Top view",
    "side": "Side view",
    "angled": "Angled view",
}

# Render mode options: "skeletons_only", "somas_only", "combined".
FIGURE_3D_RENDER_MODE = "combined"

# Plotly static output size. Bigger width/height gives nicer PDF/PNG.
FIGURE_3D_WIDTH = 1450
FIGURE_3D_HEIGHT = 1150
FIGURE_3D_SCALE = 2

# These control the same interactive 3D renderer; tune for less crowded figures.
FIGURE_3D_SHELL_ALPHA = 0.035
FIGURE_3D_CELL_LINE_WIDTH = 2.0
FIGURE_3D_CONTEXT_LINE_WIDTH = 0.35
FIGURE_3D_SHOW_CONTEXT = False
FIGURE_3D_SHOW_SOMA_TEXT = False
FIGURE_3D_MAX_CELLS = 180

# Batch rendering controls.
FIGURE_TOP_N_CLUSTERS = 12
FIGURE_MIN_CLUSTER_SIZE_TO_RENDER = 2

# UMAP style.
FIGURE_UMAP_POINT_SIZE = 34
FIGURE_UMAP_ALPHA = 0.92
FIGURE_UMAP_EDGE_LW = 0.3
FIGURE_UMAP_LABEL_CLUSTERS = True
FIGURE_UMAP_LABEL_POINTS = False

# Custom cluster defaults.
CUSTOM_DATASET_KEY = FIGURE_DATASET_KEY
CUSTOM_CLUSTER_ID = 1
CUSTOM_3D_VIEWS = ["top", "side", "angled"]
CUSTOM_RENDER_MODE_3D = "combined"
CUSTOM_SHOW_CONTEXT = False
CUSTOM_SHOW_SOMA_TEXT_IN_3D = False
CUSTOM_CELL_LINE_WIDTH = 2.0
CUSTOM_CONTEXT_LINE_WIDTH = 0.35
CUSTOM_SHELL_ALPHA = 0.035
CUSTOM_MAX_CELLS = 180

print("Fixed 3D figure-ready outputs will be saved to:", FIGURE_READY_DIR)
print("Using dataset:", FIGURE_DATASET_KEY)
print("Available datasets:", list(datasets.keys()))


In [ ]:

# ============================================================
# 12. Figure-ready helper functions — UMAPs + fixed-camera 3D exports
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from pathlib import Path
from IPython.display import display




def cluster_body_ids(ds, cluster):
    """Return body IDs in a cluster from a dataset assignment table."""
    a = ds["assignments"].copy()
    a["bodyId"] = a["bodyId"].astype(int)
    a["cluster"] = a["cluster"].astype(int)
    return a.loc[a["cluster"] == int(cluster), "bodyId"].astype(int).tolist()

# Alias used by earlier interactive rendering helpers.
_safe_cluster_ids = cluster_body_ids


def get_active_embedding_df(dataset_key=FIGURE_DATASET_KEY, method="umap"):
    """Robustly return embedding DataFrame from either dict-style or old DataFrame-style storage."""
    ds = datasets[dataset_key]
    emb_obj = ds.get("embeddings", None)
    if isinstance(emb_obj, dict):
        emb = emb_obj.get(method)
    elif isinstance(emb_obj, pd.DataFrame):
        emb = emb_obj if method == "umap" else None
    else:
        emb = None
    if emb is None and method == "umap" and "active_embedding_df" in ds:
        emb = ds["active_embedding_df"]
    return emb.copy() if emb is not None else None


def _require_fixed_3d_figure_objects():
    required = [
        "datasets",
        "render_cluster_3d_mode",
        "cluster_body_ids",
        "get_cluster_color",
        "outer_shell_vertices_render",
        "outer_shell_faces",
    ]
    missing = [x for x in required if x not in globals()]
    if missing:
        raise RuntimeError(
            "Missing required variables/functions for fixed 3D exports: "
            + ", ".join(missing)
            + ". Run the earlier loading/rendering cells first."
        )
    if not HAS_PLOTLY:
        raise RuntimeError("Plotly is required for the fixed 3D figure exports.")


def save_matplotlib_all_formats(fig, out_base, dpi=450):
    out_base = Path(out_base)
    out_base.parent.mkdir(parents=True, exist_ok=True)
    paths = []
    if SAVE_PNG:
        p = out_base.with_suffix(".png")
        fig.savefig(p, dpi=dpi, bbox_inches="tight")
        paths.append(p)
    if SAVE_PDF:
        p = out_base.with_suffix(".pdf")
        fig.savefig(p, bbox_inches="tight")
        paths.append(p)
    if SAVE_SVG:
        p = out_base.with_suffix(".svg")
        fig.savefig(p, bbox_inches="tight")
        paths.append(p)
    for p in paths:
        print("Saved:", p)
    return paths


def save_plotly_all_formats(fig, out_base, scale=FIGURE_3D_SCALE, save_html=True):
    """Save a Plotly figure as HTML plus static PNG/PDF/SVG if enabled."""
    out_base = Path(out_base)
    out_base.parent.mkdir(parents=True, exist_ok=True)
    paths = []

    if save_html and SAVE_INTERACTIVE_HTML:
        p = out_base.with_suffix(".html")
        fig.write_html(str(p))
        paths.append(p)
        print("Saved HTML:", p)

    try:
        if SAVE_PNG:
            p = out_base.with_suffix(".png")
            fig.write_image(str(p), scale=scale)
            paths.append(p)
            print("Saved PNG:", p)
        if SAVE_PDF:
            p = out_base.with_suffix(".pdf")
            fig.write_image(str(p), scale=scale)
            paths.append(p)
            print("Saved PDF:", p)
        if SAVE_SVG:
            p = out_base.with_suffix(".svg")
            fig.write_image(str(p), scale=scale)
            paths.append(p)
            print("Saved SVG:", p)
    except Exception as e:
        print(
            "Static Plotly export failed. The HTML/displayed figure is still usable.\n"
            "Install/update kaleido in this environment with:\n"
            "  pip install -U kaleido\n"
            f"Error: {repr(e)}"
        )
    return paths


def fixed_3d_camera(view):
    """Camera presets for static Plotly 3D exports."""
    view = str(view).lower()
    if view == "top":
        return dict(eye=dict(x=0.0, y=0.0, z=2.35), up=dict(x=0, y=1, z=0))
    if view == "side":
        return dict(eye=dict(x=2.35, y=0.0, z=0.0), up=dict(x=0, y=0, z=1))
    if view == "front":
        return dict(eye=dict(x=0.0, y=2.35, z=0.0), up=dict(x=0, y=0, z=1))
    if view == "angled":
        return dict(eye=dict(x=1.55, y=1.65, z=1.15), up=dict(x=0, y=0, z=1))
    raise ValueError(f"Unknown fixed 3D view: {view}")


def polish_fixed_3d_figure(fig, title, view):
    """Make the same interactive 3D brain render look like a clean static figure."""
    fig.update_layout(
        title=dict(text=title, x=0.5, xanchor="center", font=dict(size=22)),
        scene=dict(
            xaxis=dict(visible=False, showbackground=False),
            yaxis=dict(visible=False, showbackground=False),
            zaxis=dict(visible=False, showbackground=False),
            aspectmode="data",
            camera=fixed_3d_camera(view),
            bgcolor="white",
        ),
        paper_bgcolor="white",
        plot_bgcolor="white",
        margin=dict(l=0, r=0, t=70, b=0),
        width=FIGURE_3D_WIDTH,
        height=FIGURE_3D_HEIGHT,
        showlegend=False,
    )
    return fig


def _temporarily_set_render_globals(cell_line_width, context_line_width, shell_alpha, max_cells):
    """Context-manager-like helper for temporarily tuning the existing 3D renderer."""
    old = {}
    replacements = {
        "CELL_LINE_WIDTH": cell_line_width,
        "CONTEXT_LINE_WIDTH": context_line_width,
        "SHELL_ALPHA": shell_alpha,
        "MAX_CELLS_PER_CLUSTER_RENDER": max_cells,
    }
    for k, v in replacements.items():
        if k in globals() and v is not None:
            old[k] = globals()[k]
            globals()[k] = v
    return old


def _restore_render_globals(old):
    for k, v in old.items():
        globals()[k] = v


def make_fixed_3d_cluster_view(
    dataset_key=FIGURE_DATASET_KEY,
    cluster=1,
    view="angled",
    render_mode=FIGURE_3D_RENDER_MODE,
    show_context=FIGURE_3D_SHOW_CONTEXT,
    show_soma_text=FIGURE_3D_SHOW_SOMA_TEXT,
    cell_line_width=FIGURE_3D_CELL_LINE_WIDTH,
    context_line_width=FIGURE_3D_CONTEXT_LINE_WIDTH,
    shell_alpha=FIGURE_3D_SHELL_ALPHA,
    max_cells=FIGURE_3D_MAX_CELLS,
    out_dir=None,
    save_outputs=True,
    display_figure=True,
):
    """
    Make one fixed-camera 3D cluster figure using the SAME Plotly brain shell
    as the interactive renderer. This is the key fix relative to the old ugly
    2D shell projection panels.
    """
    _require_fixed_3d_figure_objects()
    if dataset_key not in datasets:
        raise ValueError(f"Unknown dataset_key={dataset_key!r}. Available: {list(datasets.keys())}")

    ds = datasets[dataset_key]
    cluster = int(cluster)
    ids = cluster_body_ids(ds, cluster)
    if len(ids) == 0:
        raise ValueError(f"No cells found for {dataset_key} cluster {cluster}")

    old = _temporarily_set_render_globals(cell_line_width, context_line_width, shell_alpha, max_cells)
    try:
        fig = render_cluster_3d_mode(
            ds,
            dataset_key,
            cluster,
            render_mode=render_mode,
            save_html=False,
            show_population_context=show_context,
            show_soma_text=show_soma_text,
        )
    finally:
        _restore_render_globals(old)

    id_preview = ", ".join(map(str, ids[:12])) + (", ..." if len(ids) > 12 else "")
    title = f"{ds['label']} | C{cluster:02d} | n={len(ids)} | {FIGURE_3D_VIEW_LABELS.get(view, view)}<br><sup>{id_preview}</sup>"
    fig = polish_fixed_3d_figure(fig, title=title, view=view)

    if display_figure:
        display(fig)

    if save_outputs:
        if out_dir is None:
            out_dir = FIG_CLUSTER_3D_DIR / dataset_key / f"C{cluster:02d}"
        out_dir = Path(out_dir)
        out_dir.mkdir(parents=True, exist_ok=True)
        out_base = out_dir / f"{dataset_key}_C{cluster:02d}_{render_mode}_{view}_fixed3D"
        save_plotly_all_formats(fig, out_base, scale=FIGURE_3D_SCALE, save_html=True)

    return fig


def make_fixed_3d_cluster_views(
    dataset_key=FIGURE_DATASET_KEY,
    cluster=1,
    views=None,
    render_mode=FIGURE_3D_RENDER_MODE,
    show_context=FIGURE_3D_SHOW_CONTEXT,
    show_soma_text=FIGURE_3D_SHOW_SOMA_TEXT,
    cell_line_width=FIGURE_3D_CELL_LINE_WIDTH,
    context_line_width=FIGURE_3D_CONTEXT_LINE_WIDTH,
    shell_alpha=FIGURE_3D_SHELL_ALPHA,
    max_cells=FIGURE_3D_MAX_CELLS,
    out_dir=None,
    save_outputs=True,
    display_figures=True,
):
    """Make top/side/angled fixed-camera 3D figures and save each as PDF/PNG/SVG/HTML."""
    if views is None:
        views = FIGURE_3D_VIEWS

    ds = datasets[dataset_key]
    cluster = int(cluster)
    ids = cluster_body_ids(ds, cluster)

    if out_dir is None:
        out_dir = FIG_CLUSTER_3D_DIR / dataset_key / f"C{cluster:02d}"
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    id_table = pd.DataFrame({
        "dataset_key": dataset_key,
        "dataset_label": ds["label"],
        "cluster": cluster,
        "cluster_name": f"{dataset_key}_C{cluster:02d}",
        "n_cells": len(ids),
        "bodyId": ids,
    })
    id_csv = out_dir / f"{dataset_key}_C{cluster:02d}_cell_ids.csv"
    id_txt = out_dir / f"{dataset_key}_C{cluster:02d}_cell_ids.txt"
    id_table.to_csv(id_csv, index=False)
    id_txt.write_text("\n".join(map(str, ids)) + "\n")
    print("Saved cell ID table:", id_csv)
    print("Saved cell ID text:", id_txt)

    figs = {}
    for view in views:
        print("=" * 80)
        print(f"Rendering fixed 3D {view} view | {dataset_key} C{cluster:02d}")
        print("=" * 80)
        figs[view] = make_fixed_3d_cluster_view(
            dataset_key=dataset_key,
            cluster=cluster,
            view=view,
            render_mode=render_mode,
            show_context=show_context,
            show_soma_text=show_soma_text,
            cell_line_width=cell_line_width,
            context_line_width=context_line_width,
            shell_alpha=shell_alpha,
            max_cells=max_cells,
            out_dir=out_dir,
            save_outputs=save_outputs,
            display_figure=display_figures,
        )

    display(id_table)
    return figs, id_table


def plot_embedding_figure_ready(dataset_key=FIGURE_DATASET_KEY, method="umap", label_clusters=FIGURE_UMAP_LABEL_CLUSTERS, label_points=FIGURE_UMAP_LABEL_POINTS):
    """Clean UMAP plot with stable cluster colors. Robust to active reclustering storage."""
    if dataset_key not in datasets:
        print(f"Skipping {dataset_key}: dataset not loaded.")
        return None
    ds = datasets[dataset_key]
    emb = get_active_embedding_df(dataset_key, method)
    if emb is None or len(emb) == 0:
        print(f"Skipping {dataset_key} {method}: embedding not available.")
        return None

    a = ds["assignments"].copy()
    a["bodyId"] = a["bodyId"].astype(int)
    emb = emb.copy()
    emb["bodyId"] = emb["bodyId"].astype(int)
    df = emb.merge(a[["bodyId", "cluster", "cluster_name", "cluster_size"]], on="bodyId", how="left")
    df = df.dropna(subset=["cluster"])
    df["cluster"] = df["cluster"].astype(int)

    xcol = f"{method}1" if f"{method}1" in df.columns else "umap1"
    ycol = f"{method}2" if f"{method}2" in df.columns else "umap2"
    if xcol not in df.columns or ycol not in df.columns:
        print(f"Skipping {dataset_key} {method}: could not find coordinate columns.")
        return None

    fig, ax = plt.subplots(figsize=(5.2, 4.45), dpi=450)
    for cl, sub in df.groupby("cluster", sort=True):
        ax.scatter(
            sub[xcol], sub[ycol],
            s=FIGURE_UMAP_POINT_SIZE,
            alpha=FIGURE_UMAP_ALPHA,
            c=get_cluster_color(dataset_key, int(cl)),
            edgecolors="black",
            linewidths=FIGURE_UMAP_EDGE_LW,
            label=f"C{int(cl):02d} (n={len(sub)})",
        )
        if label_clusters:
            cx, cy = sub[xcol].median(), sub[ycol].median()
            ax.text(cx, cy, f"C{int(cl):02d}", fontsize=7, weight="bold", ha="center", va="center")

    if label_points:
        for _, r in df.iterrows():
            ax.text(r[xcol], r[ycol], str(int(r["bodyId"])), fontsize=4.5, alpha=0.75)

    ax.set_title(f"{ds['label']} — {method.upper()} by morphology cluster")
    ax.set_xlabel(f"{method.upper()} 1")
    ax.set_ylabel(f"{method.upper()} 2")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(False)

    # Keep legend out of the way for many clusters.
    if df["cluster"].nunique() <= 18:
        ax.legend(frameon=False, fontsize=6, loc="center left", bbox_to_anchor=(1.02, 0.5))

    out_base = FIG_UMAP_DIR / f"{dataset_key}_{method}_clusters_figure_ready"
    save_matplotlib_all_formats(fig, out_base, dpi=450)
    plt.show()
    return fig, df


In [ ]:
# ============================================================
# 13. Publication-style UMAP plot — active soma-in-ROI clustering
# ============================================================
# This uses the active UMAP selected in Cell 2b.
# MDS is intentionally skipped because the active exploration uses UMAPs
# recomputed directly from the cached NBLAST distance.

if "plot_embedding_figure_ready" in globals():
    soma_in_roi_umap_result = plot_embedding_figure_ready(FIGURE_DATASET_KEY, "umap")
else:
    soma_in_roi_umap_result = plot_active_umap_from_dataset(FIGURE_DATASET_KEY, label_points=False)


In [ ]:

# ============================================================
# 14. Batch-render top soma-in-ROI clusters as fixed 3D PDF/PNG/SVG views
# ============================================================

def render_top_fixed_3d_cluster_views(dataset_key=FIGURE_DATASET_KEY):
    """Render the largest clusters for the soma-in-Raphe-superior dataset."""
    if dataset_key not in datasets:
        print(f"{dataset_key} dataset not loaded.")
        return []

    ds = datasets[dataset_key]
    sizes = ds["assignments"]["cluster"].value_counts().sort_values(ascending=False)
    sizes = sizes.loc[sizes >= FIGURE_MIN_CLUSTER_SIZE_TO_RENDER]
    clusters = [int(c) for c in sizes.head(FIGURE_TOP_N_CLUSTERS).index]

    print(f"Rendering top {len(clusters)} fixed 3D cluster panels for {dataset_key}:")
    display(sizes.head(FIGURE_TOP_N_CLUSTERS).to_frame("n_cells"))

    outputs = []
    for cl in clusters:
        out_dir = FIG_CLUSTER_3D_DIR / dataset_key / f"C{cl:02d}"
        figs, tab = make_fixed_3d_cluster_views(
            dataset_key=dataset_key,
            cluster=cl,
            views=FIGURE_3D_VIEWS,
            render_mode=FIGURE_3D_RENDER_MODE,
            show_context=FIGURE_3D_SHOW_CONTEXT,
            show_soma_text=FIGURE_3D_SHOW_SOMA_TEXT,
            cell_line_width=FIGURE_3D_CELL_LINE_WIDTH,
            context_line_width=FIGURE_3D_CONTEXT_LINE_WIDTH,
            shell_alpha=FIGURE_3D_SHELL_ALPHA,
            max_cells=FIGURE_3D_MAX_CELLS,
            out_dir=out_dir,
            save_outputs=True,
            display_figures=False,
        )
        outputs.append(out_dir)
    return outputs

# Run batch export.
soma_in_roi_fixed_3d_outputs = render_top_fixed_3d_cluster_views(FIGURE_DATASET_KEY)


In [ ]:

# ============================================================
# 15. Custom figure-ready fixed 3D plot for one cluster
# ============================================================
# Edit these parameters and rerun this cell.

CUSTOM_DATASET_KEY = FIGURE_DATASET_KEY
CUSTOM_CLUSTER_ID = 1
CUSTOM_3D_VIEWS = ["top", "side", "angled"]
CUSTOM_RENDER_MODE_3D = "combined"  # "skeletons_only", "somas_only", or "combined"
CUSTOM_SHOW_CONTEXT = False
CUSTOM_SHOW_SOMA_TEXT_IN_3D = False
CUSTOM_CELL_LINE_WIDTH = 2.0
CUSTOM_CONTEXT_LINE_WIDTH = 0.35
CUSTOM_SHELL_ALPHA = 0.035
CUSTOM_MAX_CELLS = 180


def make_custom_cluster_fixed_3d_figure_ready_plot(
    dataset_key=CUSTOM_DATASET_KEY,
    cluster=CUSTOM_CLUSTER_ID,
    views=CUSTOM_3D_VIEWS,
    render_mode_3d=CUSTOM_RENDER_MODE_3D,
    show_context=CUSTOM_SHOW_CONTEXT,
    show_soma_text_3d=CUSTOM_SHOW_SOMA_TEXT_IN_3D,
    cell_line_width=CUSTOM_CELL_LINE_WIDTH,
    context_line_width=CUSTOM_CONTEXT_LINE_WIDTH,
    shell_alpha=CUSTOM_SHELL_ALPHA,
    max_cells=CUSTOM_MAX_CELLS,
):
    """
    Create figure-ready fixed-camera 3D PDFs/PNGs/SVGs for one cluster.

    Unlike the old projection panel, this uses the same nice interactive 3D
    outer brain shell, just with fixed cameras for static export.
    """
    if dataset_key not in datasets:
        raise ValueError(f"Unknown dataset_key={dataset_key!r}. Available: {list(datasets.keys())}")

    ds = datasets[dataset_key]
    cluster = int(cluster)
    ids = cluster_body_ids(ds, cluster)
    if len(ids) == 0:
        raise ValueError(f"No cells found for {dataset_key} cluster {cluster}")

    custom_dir = FIG_SINGLE_CLUSTER_DIR / dataset_key / f"C{cluster:02d}"
    custom_dir.mkdir(parents=True, exist_ok=True)

    print(f"{ds['label']} | C{cluster:02d} | n={len(ids)}")
    display(pd.DataFrame({"bodyId": ids}))

    figs, id_table = make_fixed_3d_cluster_views(
        dataset_key=dataset_key,
        cluster=cluster,
        views=views,
        render_mode=render_mode_3d,
        show_context=show_context,
        show_soma_text=show_soma_text_3d,
        cell_line_width=cell_line_width,
        context_line_width=context_line_width,
        shell_alpha=shell_alpha,
        max_cells=max_cells,
        out_dir=custom_dir,
        save_outputs=True,
        display_figures=True,
    )

    return figs, id_table


custom_fixed_3d_figs, custom_fixed_3d_cell_table = make_custom_cluster_fixed_3d_figure_ready_plot()
